# 🎮 Steam — Analyse globale du marché du jeu vidéo

**Client (fictif)** : Ubisoft  ·  **Formation** : Jedha — Certification Data Science & Data Engineering
**Source** : `s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json` (extraction de l'API SteamSpy)
**Stack** : Databricks · PySpark · visualisations natives Databricks

---

## Objectif

Comprendre **quels facteurs influencent la popularité et les ventes d'un jeu vidéo**, et cartographier
le marché Steam pour éclairer le lancement d'un nouveau titre.

L'analyse se déroule à trois niveaux : **macro** (marché global), **genres**, **plateformes**,
puis une quatrième partie qui croise les variables pour répondre à la question centrale.

---

## ⚠️ Périmètre et limites — à lire avant les résultats

| Point | Ce qu'il faut savoir |
|---|---|
| **Devise** | SteamSpy exprime les prix en **centimes de dollar US**. `price = 999` signifie **9,99 $**. Toutes les valeurs affichées dans ce notebook sont converties en **dollars** (`price_usd`). |
| **Typage** | `price`, `initialprice`, `discount`, `required_age` sont stockés en **chaînes de caractères**. Sans cast explicite, `min()` et `max()` comparent lexicographiquement (`"12499" < "9999"`) et renvoient des résultats faux. Tous ces champs sont castés en partie 2. |
| **Valeurs manquantes** | Elles sont encodées en **chaînes vides `""`**, pas en `NULL`. Un `isNull()` naïf conclurait à tort que le dataset est complet. Elles sont normalisées en `NULL` en partie 2. |
| **Périmètre du catalogue** | Le dataset contient des **jeux, mais aussi des logiciels** vendus sur Steam (Aseprite, suites audio/vidéo, utilitaires…). Le champ `type` **ne suffit pas** à les distinguer : il vaut `game` pour 55 690 des 55 691 entrées, logiciels compris. Le tri se fait donc sur le **genre déclaré** (partie 2.6) — Steam range ces applications dans des genres qui n'existent pas pour un jeu. On travaille sur `df_games` (jeux uniquement) pour les analyses de marché, et sur `df_clean` (catalogue complet) pour les analyses de catalogue. Chaque section précise le périmètre utilisé. |
| **Genres multiples** | Un jeu porte en moyenne 2 à 4 genres. Après `explode`, la somme des jeux par genre **dépasse largement** le total du catalogue. Les comptages par genre sont des **occurrences**, jamais des parts de marché directes. |
| **Ventes** | Le dataset ne contient **pas** de chiffre de ventes. `owners` fournit une **fourchette** de possesseurs (ex. `"1,000,000 .. 2,000,000"`) : on utilise le milieu de fourchette comme proxy, en gardant à l'esprit que la fourchette est large et que le prix actuel n'est pas le prix de vente historique. |
| **Avis ≠ ventes** | Le nombre d'avis n'est pas un nombre de ventes (ratio usuel de l'ordre de 30 à 50×). Il est utilisé comme indicateur de **visibilité / engagement**, pas de chiffre d'affaires. |
| **Photographie instantanée** | Les avis, prix, promotions et joueurs simultanés correspondent à la date d'extraction du dataset, pas à l'historique du jeu. |

---

## Plan du notebook

1. **Chargement** des données et lecture du schéma imbriqué
2. **Préparation** : aplatissement, typage, nettoyage, contrôles qualité
3. **Analyse macro** : éditeurs, qualité, temporalité (dont Covid), prix, promotions, langues, classification par âge, fonctionnalités
4. **Analyse par genre** : représentation, qualité pondérée, prix, revenu estimé, spécialités des éditeurs
5. **Analyse par plateforme** : répartition, combinaisons, qualité, genres préférentiels
6. **Facteurs de succès** : corrélations et analyses croisées — la réponse à la question centrale
7. **Synthèse chiffrée** (générée depuis les données) et **recommandations métier**

---
# 1. Chargement des données

In [0]:
# Convention d'import : on n'utilise JAMAIS `from pyspark.sql.functions import *`.
# Cet import masque les fonctions natives Python (round, sum, min, max, abs, filter)
# et oblige ensuite à des contournements fragiles du type `__builtins__.round(...)`.
# Le préfixe F garde les deux espaces de noms disponibles.

from pyspark.sql import functions as F
from pyspark.sql import Row

# Le champ `release_date` mélange plusieurs formats. En politique par défaut
# (EXCEPTION), Spark 3 lève une SparkUpgradeException dès qu'une valeur ne
# correspond pas au motif demandé. En CORRECTED, il renvoie simplement NULL,
# ce qui permet d'enchaîner plusieurs formats avec coalesce (cf. 2.3).
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

print("Imports OK — fonctions PySpark accessibles via le préfixe F.")
print("round(), sum(), min(), max() natifs Python restent disponibles.")

Imports OK — fonctions PySpark accessibles via le préfixe F.
round(), sum(), min(), max() natifs Python restent disponibles.


In [0]:
S3_PATH = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

df_raw = spark.read.json(S3_PATH)

# Une seule action pour compter : la valeur est stockée et réutilisée ensuite.
NB_LIGNES_BRUTES = df_raw.count()

print(f"Lignes brutes chargées : {NB_LIGNES_BRUTES:,}")

Lignes brutes chargées : 55,691


In [0]:
# `data.tags` est une struct de plusieurs centaines de colonnes : un printSchema() complet
# produit ~23 000 caractères illisibles. On n'affiche que le premier niveau.

print("Champs de premier niveau :", df_raw.columns)
print(f"\nChamps disponibles dans la struct `data` ({len(df_raw.schema['data'].dataType.fields)}) :\n")

for field in df_raw.schema["data"].dataType.fields:
    type_court = field.dataType.simpleString()
    if len(type_court) > 45:
        type_court = type_court[:45] + "…"
    print(f"  - {field.name:<20} {type_court}")

Champs de premier niveau : ['data', 'id']

Champs disponibles dans la struct `data` (22) :

  - appid                bigint
  - categories           array<string>
  - ccu                  bigint
  - developer            string
  - discount             string
  - genre                string
  - header_image         string
  - initialprice         string
  - languages            string
  - name                 string
  - negative             bigint
  - owners               string
  - platforms            struct<linux:boolean,mac:boolean,windows:bool…
  - positive             bigint
  - price                string
  - publisher            string
  - release_date         string
  - required_age         string
  - short_description    string
  - tags                 struct<1980s:bigint,1990's:bigint,2.5D:bigint…
  - type                 string
  - website              string


In [0]:
# Aperçu brut avant tout traitement.
# On sélectionne quelques champs plutôt que la struct entière : `data.tags`
# contient plusieurs centaines de colonnes et rend l'affichage illisible.
display(
    df_raw.select(
        "id",
        F.col("data.appid").alias("appid"),
        F.col("data.name").alias("name"),
        F.col("data.type").alias("type"),
        F.col("data.release_date").alias("release_date"),
        F.col("data.price").alias("price"),
        F.col("data.owners").alias("owners"),
        F.col("data.genre").alias("genre"),
        F.col("data.platforms").alias("platforms"),
    ).limit(10)
)

id,appid,name,type,release_date,price,owners,genre,platforms
10,10,Counter-Strike,game,2000/11/1,999,"10,000,000 .. 20,000,000",Action,"List(true, true, true)"
1000000,1000000,ASCENXION,game,2021/05/14,999,"0 .. 20,000","Action, Adventure, Indie","List(false, false, true)"
1000010,1000010,Crown Trick,game,2020/10/16,599,"200,000 .. 500,000","Adventure, Indie, RPG, Strategy","List(false, false, true)"
1000030,1000030,"Cook, Serve, Delicious! 3?!",game,2020/10/14,1999,"100,000 .. 200,000","Action, Indie, Simulation, Strategy","List(false, true, true)"
1000040,1000040,细胞战争,game,2019/03/30,199,"0 .. 20,000","Action, Casual, Indie, Simulation","List(false, false, true)"
1000080,1000080,Zengeon,game,2019/06/24,799,"100,000 .. 200,000","Action, Adventure, Indie, RPG","List(false, true, true)"
1000100,1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,game,2019/01/24,1299,"0 .. 20,000","Adventure, Indie, RPG, Strategy","List(false, false, true)"
1000110,1000110,Jumping Master(跳跳大咖),game,2019/04/8,0,"20,000 .. 50,000","Action, Adventure, Casual, Free to Play, Massively Multiplayer","List(false, false, true)"
1000130,1000130,Cube Defender,game,2019/01/6,299,"0 .. 20,000","Casual, Indie","List(false, true, true)"
1000280,1000280,Tower of Origin2-Worm's Nest,game,2021/09/9,1399,"0 .. 20,000","Indie, RPG","List(false, false, true)"


---
# 2. Préparation des données

Cette partie conditionne la validité de tout ce qui suit. Cinq opérations :

1. **Aplatir** la struct `data` — en récupérant *tous* les champs utiles, y compris `release_date`,
   `required_age`, `type` et `owners`, indispensables pour répondre au cahier des charges.
2. **Normaliser** les chaînes vides en `NULL`.
3. **Typer** les colonnes numériques stockées en texte, et convertir les prix en dollars.
4. **Dériver** les variables d'analyse (ratio d'avis, score de Wilson, possesseurs, nombre de langues…).
5. **Contrôler** la qualité : unicité de la clé, valeurs manquantes réelles, périmètre du catalogue.

In [0]:
# --- 2.1 Aplatissement de la structure imbriquée -------------------------------
# `platforms` est aplati ICI, une fois pour toutes : aucune cellule en aval ne
# retouche `data.platforms`.

df_clean = df_raw.select(
    F.col("id"),
    F.col("data.appid").alias("appid"),
    F.col("data.name").alias("name"),
    F.col("data.type").alias("type"),
    F.col("data.developer").alias("developer"),
    F.col("data.publisher").alias("publisher"),
    F.col("data.release_date").alias("release_date"),
    F.col("data.required_age").alias("required_age"),
    F.col("data.positive").alias("positive"),
    F.col("data.negative").alias("negative"),
    F.col("data.owners").alias("owners"),
    F.col("data.ccu").alias("ccu"),
    F.col("data.price").alias("price"),
    F.col("data.initialprice").alias("initialprice"),
    F.col("data.discount").alias("discount"),
    F.col("data.languages").alias("languages"),
    F.col("data.genre").alias("genre"),
    F.col("data.categories").alias("categories"),
    F.col("data.short_description").alias("short_description"),
    F.col("data.header_image").alias("header_image"),
    F.col("data.platforms.windows").alias("windows"),
    F.col("data.platforms.mac").alias("mac"),
    F.col("data.platforms.linux").alias("linux"),
)

print(f"{len(df_clean.columns)} colonnes extraites (la struct `tags`, non exploitée, est volontairement écartée).")
print(df_clean.columns)

23 colonnes extraites (la struct `tags`, non exploitée, est volontairement écartée).
['id', 'appid', 'name', 'type', 'developer', 'publisher', 'release_date', 'required_age', 'positive', 'negative', 'owners', 'ccu', 'price', 'initialprice', 'discount', 'languages', 'genre', 'categories', 'short_description', 'header_image', 'windows', 'mac', 'linux']


In [0]:
# --- 2.2 Normalisation des valeurs manquantes ---------------------------------
# Dans ce dataset, l'absence de valeur est encodée par une CHAÎNE VIDE, pas par NULL.
# Sans cette normalisation, isNull() conclut à tort que le dataset est complet,
# et un éditeur vide se retrouve dans le top 10 des éditeurs.

colonnes_texte = [c for c, t in df_clean.dtypes if t == "string"]

df_clean = df_clean.select(*[
    (F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c)).alias(c)
     if c in colonnes_texte else F.col(c))
    for c in df_clean.columns
])

print(f'{len(colonnes_texte)} colonnes texte normalisées ("" -> NULL) :')
print(colonnes_texte)

15 colonnes texte normalisées ("" -> NULL) :
['id', 'name', 'type', 'developer', 'publisher', 'release_date', 'required_age', 'owners', 'price', 'initialprice', 'discount', 'languages', 'genre', 'short_description', 'header_image']


In [0]:
# --- 2.3 Typage et variables dérivées -----------------------------------------

Z = 1.96  # niveau de confiance 95 % pour le score de Wilson

n_reviews = F.col("positive") + F.col("negative")
p_positif = F.col("positive") / n_reviews
# Borne basse de l'intervalle de confiance de Wilson : pénalise les jeux à faible
# volume d'avis, ce qui évite qu'un jeu à 104 avis et 100 % domine le classement.
wilson = (
    (p_positif + F.lit(Z * Z) / (2 * n_reviews)
     - F.lit(Z) * F.sqrt((p_positif * (1 - p_positif) + F.lit(Z * Z) / (4 * n_reviews)) / n_reviews))
    / (1 + F.lit(Z * Z) / n_reviews)
)

df_clean = (
    df_clean

    # --- Prix : SteamSpy les exprime en CENTIMES de dollar US, en colonnes STRING.
    #     Sans ce cast, avg() renvoie 773 au lieu de 7,73 et min()/max() sont lexicographiques.
    .withColumn("price_usd",        F.col("price").cast("double") / 100)
    .withColumn("initialprice_usd", F.col("initialprice").cast("double") / 100)
    .withColumn("discount_pct",     F.col("discount").cast("double"))

    # --- Âge requis : "0", "18", parfois "18+" -> on extrait le premier entier rencontré.
    .withColumn("required_age_int",
                F.when(F.regexp_extract(F.coalesce(F.col("required_age"), F.lit("")), r"(\d+)", 1) != "",
                       F.regexp_extract(F.coalesce(F.col("required_age"), F.lit("")), r"(\d+)", 1).cast("int")))

    # --- Possesseurs : "1,000,000 .. 2,000,000" -> bornes et milieu de fourchette.
    .withColumn("owners_clean", F.regexp_replace(F.coalesce(F.col("owners"), F.lit("")), ",", ""))
    .withColumn("owners_min",   F.regexp_extract(F.col("owners_clean"), r"^(\d+)", 1).cast("long"))
    .withColumn("owners_max",   F.regexp_extract(F.col("owners_clean"), r"(\d+)\s*$", 1).cast("long"))
    .withColumn("owners_mid",   (F.col("owners_min") + F.col("owners_max")) / 2.0)

    # --- Date de sortie : l'année par regex (robuste à tous les formats),
    #     la date complète par coalesce sur les formats rencontrés chez SteamSpy.
    .withColumn("release_year",
                F.when(F.regexp_extract(F.coalesce(F.col("release_date"), F.lit("")), r"(19|20)\d{2}", 0) != "",
                       F.regexp_extract(F.coalesce(F.col("release_date"), F.lit("")), r"(19|20)\d{2}", 0).cast("int")))
    .withColumn("release_dt", F.coalesce(
        F.try_to_date(F.col("release_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("release_date"), "MMM d, yyyy"),
        F.try_to_date(F.col("release_date"), "d MMM, yyyy"),
        F.try_to_date(F.col("release_date"), "MMM yyyy"),
        F.try_to_date(F.col("release_date"), "yyyy/MM/dd"),
        F.try_to_date(F.col("release_date"), "yyyy/M/d"),
    ))
    .withColumn("release_month", F.date_format(F.col("release_dt"), "yyyy-MM"))

    # --- Avis : total, ratio brut, et score de Wilson.
    .withColumn("total_reviews", n_reviews)
    .withColumn("ratio_positif", F.when(n_reviews > 0, F.round(p_positif * 100, 2)))
    .withColumn("score_wilson",  F.when(n_reviews > 0, F.round(wilson * 100, 2)))

    # --- Plateformes : nombre de systèmes supportés.
    .withColumn("nb_plateformes",
                F.coalesce(F.col("windows").cast("int"), F.lit(0))
                + F.coalesce(F.col("mac").cast("int"), F.lit(0))
                + F.coalesce(F.col("linux").cast("int"), F.lit(0)))

    # --- Langues : SteamSpy laisse des balises HTML, des astérisques et une mention
    #     de fin ("languages with full audio support") dans le champ.
    .withColumn("languages_clean",
                F.trim(F.regexp_replace(
                    F.regexp_replace(F.coalesce(F.col("languages"), F.lit("")),
                                     r"<[^>]+>|\*|languages with full audio support", ""),
                    r"^[,\s]+|[,\s]+$", "")))
    .withColumn("nb_langues",
                F.when(F.col("languages_clean") != "", F.size(F.split(F.col("languages_clean"), ","))))

    # --- Catégories Steam : nombre de fonctionnalités déclarées.
    .withColumn("nb_categories",
                F.when(F.col("categories").isNotNull(), F.size(F.col("categories"))).otherwise(0))

    # --- Engagement : part des possesseurs qui laissent un avis.
    .withColumn("taux_avis",
                F.when(F.col("owners_mid") > 0, F.col("total_reviews") / F.col("owners_mid")))
)

print("Colonnes dérivées ajoutées :")
print("  prix       -> price_usd, initialprice_usd, discount_pct")
print("  âge        -> required_age_int")
print("  ventes     -> owners_min, owners_max, owners_mid")
print("  temps      -> release_year, release_dt, release_month")
print("  qualité    -> total_reviews, ratio_positif, score_wilson")
print("  richesse   -> nb_plateformes, nb_langues, nb_categories")
print("  engagement -> taux_avis")

Colonnes dérivées ajoutées :
  prix       -> price_usd, initialprice_usd, discount_pct
  âge        -> required_age_int
  ventes     -> owners_min, owners_max, owners_mid
  temps      -> release_year, release_dt, release_month
  qualité    -> total_reviews, ratio_positif, score_wilson
  richesse   -> nb_plateformes, nb_langues, nb_categories
  engagement -> taux_avis


In [0]:
# --- 2.4 Contrôle d'unicité ----------------------------------
# Note : cache() n'est pas disponible sur Serverless.
# Les résultats intermédiaires sont automatiquement optimisés par Spark Connect.

NB_LIGNES      = df_clean.count()
nb_appid_uniq  = df_clean.select("appid").distinct().count()
nb_noms_uniq   = df_clean.select("name").distinct().count()

print(f"Lignes                : {NB_LIGNES:,}")
print(f"appid distincts       : {nb_appid_uniq:,}")
print(f"Noms distincts        : {nb_noms_uniq:,}  ({NB_LIGNES - nb_noms_uniq:,} noms en doublon)")

if nb_appid_uniq < NB_LIGNES:
    print(f"\n⚠️  {NB_LIGNES - nb_appid_uniq:,} doublons sur la clé appid -> déduplication.")
    df_clean = df_clean.dropDuplicates(["appid"])
    NB_LIGNES = df_clean.count()
    print(f"Lignes après déduplication : {NB_LIGNES:,}")
else:
    print("\n✅ `appid` est unique : aucune déduplication nécessaire.")
    print("   Les noms en doublon correspondent à des applications distinctes")
    print("   (éditions, bundles, rééditions, homonymes) et sont conservés.")

Lignes                : 55,691
appid distincts       : 55,691
Noms distincts        : 55,430  (261 noms en doublon)

✅ `appid` est unique : aucune déduplication nécessaire.
   Les noms en doublon correspondent à des applications distinctes
   (éditions, bundles, rééditions, homonymes) et sont conservés.


In [0]:
# --- 2.5 Rapport de valeurs manquantes ----------------------------------------
# On compte les NULL *et* les chaînes vides résiduelles : c'est la seule façon
# d'obtenir un état réel de la complétude sur ce dataset.

def rapport_valeurs_manquantes(df, nb_lignes):
    exprs = []
    for c, t in df.dtypes:
        condition = F.col(c).isNull()
        if t == "string":
            condition = condition | (F.trim(F.col(c)) == "")
        exprs.append(F.sum(F.when(condition, 1).otherwise(0)).alias(c))
    resultat = df.select(*exprs).collect()[0].asDict()
    lignes = [Row(colonne=k,
                  nb_manquants=int(v),
                  pct_manquants=round(100.0 * v / nb_lignes, 2))
              for k, v in resultat.items()]
    return spark.createDataFrame(lignes).orderBy(F.desc("nb_manquants"))

df_missing = rapport_valeurs_manquantes(df_clean, NB_LIGNES)
display(df_missing)

colonne,nb_manquants,pct_manquants
release_dt,222,0.4
release_month,222,0.4
ratio_positif,163,0.29
score_wilson,163,0.29
genre,161,0.29
publisher,134,0.24
developer,127,0.23
release_date,99,0.18
release_year,99,0.18
short_description,37,0.07


In [0]:
# --- 2.6 Périmètre : le catalogue n'est pas composé que de jeux ----------------
# Deux filtres complémentaires, parce qu'ils ne captent pas la même chose.

# (a) Le champ `type`. Valeurs attendues : game / dlc / demo / software.
display(
    df_clean.groupBy("type")
            .agg(F.count("*").alias("nombre"))
            .orderBy(F.desc("nombre"))
)

# (b) Le genre déclaré. Sur CE dataset, `type` vaut "game" pour la quasi-totalité
#     des entrées — LOGICIELS COMPRIS (Aseprite, suites audio/vidéo, utilitaires).
#     Le champ `type` est donc inopérant pour isoler les jeux, et le seul signal
#     fiable est le genre : Steam range ces applications dans des catégories qui
#     n'existent pas pour un jeu.
GENRES_LOGICIELS = [
    "Utilities", "Design & Illustration", "Animation & Modeling",
    "Audio Production", "Video Production", "Photo Editing",
    "Web Publishing", "Software Training", "Game Development",
    "Education", "Accounting", "Movie",
]

df_clean = (
    df_clean
    .withColumn(
        "genres_array",
        F.expr("filter(transform(split(coalesce(genre, ''), ','), g -> trim(g)), g -> g != '')"),
    )
    .withColumn(
        "est_logiciel",
        # Logiciel = déclare au moins un genre non ludique ET aucun genre de jeu.
        # Un jeu éducatif ("Casual, Education") reste donc bien un jeu.
        (F.size(F.array_intersect(
            F.col("genres_array"), F.array(*[F.lit(g) for g in GENRES_LOGICIELS]))) > 0)
        & (F.size(F.array_except(
            F.col("genres_array"), F.array(*[F.lit(g) for g in GENRES_LOGICIELS]))) == 0),
    )
)

types_presents = [r["type"] for r in df_clean.select("type").distinct().collect()]
filtre_type = (F.col("type") == "game") if "game" in types_presents else F.lit(True)

df_games = df_clean.filter(filtre_type & (~F.col("est_logiciel")))

# Note : cache() n'est pas disponible sur Serverless.
NB_LIGNES     = df_clean.count()
NB_JEUX       = df_games.count()
nb_logiciels  = df_clean.filter(F.col("est_logiciel")).count()
nb_hors_type  = df_clean.filter(~filtre_type).count()

print(f"\nCatalogue complet (df_clean)  : {NB_LIGNES:,} entrées")
print(f"  – écartées par le champ `type` : {nb_hors_type:,}")
print(f"  – écartées comme logiciels     : {nb_logiciels:,}  (genre non ludique uniquement)")
print(f"Jeux uniquement (df_games)    : {NB_JEUX:,} entrées  ({100.0 * NB_JEUX / NB_LIGNES:.1f} %)")

print("\nExemples d'entrées écartées comme logiciels :")
display(
    df_clean.filter(F.col("est_logiciel"))
            .select("name", "genre", "price_usd", "positive")
            .orderBy(F.desc("positive"))
            .limit(10)
)

print("\nConvention retenue pour la suite du notebook :")
print("  • df_games -> analyses de MARCHÉ du jeu vidéo (prix, genres, qualité, plateformes)")
print("  • df_clean -> analyses de CATALOGUE Steam (volume global, langues, fonctionnalités)")

type,nombre
game,55690
hardware,1



Catalogue complet (df_clean)  : 55,691 entrées
  – écartées par le champ `type` : 1
  – écartées comme logiciels     : 965  (genre non ludique uniquement)
Jeux uniquement (df_games)    : 54,725 entrées  (98.3 %)

Exemples d'entrées écartées comme logiciels :


name,genre,price_usd,positive
Soundpad,"Audio Production, Utilities",4.99,44518
Blender,"Animation & Modeling, Design & Illustration, Video Production",0.0,36892
Source Filmmaker,"Animation & Modeling, Video Production",0.0,28871
Aseprite,"Animation & Modeling, Design & Illustration, Game Development",19.99,11823
3DMark,Utilities,34.99,11192
CPUCores :: Maximize Your FPS,"Design & Illustration, Utilities",7.49,8438
Easy eSports,Utilities,0.0,8046
FaceRig,"Animation & Modeling, Video Production",0.0,6880
RPG Maker MV,"Design & Illustration, Web Publishing",79.99,5966
Action! - Gameplay Recording and Streaming,"Audio Production, Education, Software Training, Utilities, Video Production, Web Publishing",29.99,5098



Convention retenue pour la suite du notebook :
  • df_games -> analyses de MARCHÉ du jeu vidéo (prix, genres, qualité, plateformes)
  • df_clean -> analyses de CATALOGUE Steam (volume global, langues, fonctionnalités)


In [0]:
# --- 2.7 Aperçu du dataset préparé --------------------------------------------
display(
    df_games.select(
        "appid", "name", "publisher", "release_year", "type",
        "price_usd", "discount_pct", "owners_mid",
        "positive", "negative", "ratio_positif", "score_wilson",
        "required_age_int", "nb_langues", "nb_plateformes",
    ).limit(20)
)

appid,name,publisher,release_year,type,price_usd,discount_pct,owners_mid,positive,negative,ratio_positif,score_wilson,required_age_int,nb_langues,nb_plateformes
10,Counter-Strike,Valve,2000,game,9.99,0.0,1.5E7,201215,5199,97.48,97.41,0,8,3
1000000,ASCENXION,PsychoFlux Entertainment,2021,game,9.99,0.0,10000.0,27,5,84.38,68.25,0,3,1
1000010,Crown Trick,"Team17, NEXT Studios",2020,game,5.99,70.0,350000.0,4032,646,86.19,85.17,0,9,1
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,2020,game,19.99,0.0,150000.0,1575,115,93.2,91.89,0,1,2
1000040,细胞战争,DoubleC Games,2019,game,1.99,0.0,10000.0,0,1,0.0,0.0,0,1,1
1000080,Zengeon,2P Games,2019,game,7.99,60.0,150000.0,1018,462,68.78,66.38,0,5,2
1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,Starship Studio,2019,game,12.99,0.0,10000.0,18,6,75.0,55.1,0,3,1
1000110,Jumping Master(跳跳大咖),重庆环游者网络科技,2019,game,0.0,0.0,35000.0,50,34,59.52,48.83,0,3,1
1000130,Cube Defender,Simon Codrington,2019,game,2.99,0.0,10000.0,6,0,100.0,60.97,0,1,2
1000280,Tower of Origin2-Worm's Nest,Villain Role,2021,game,13.99,0.0,10000.0,32,12,72.73,58.15,0,3,1


---
# 3. Analyse macro — vue d'ensemble du marché

*Périmètre : `df_games` (jeux uniquement), sauf mention contraire.*

## 3.1 Quel éditeur a publié le plus de jeux ?

On mesure le volume, mais aussi la **qualité pondérée** : un éditeur qui publie 400 jeux
moyens et un éditeur qui en publie 100 excellents ne racontent pas la même histoire.

In [0]:
df_publishers = (
    df_games
    .filter(F.col("publisher").isNotNull())      # NULL réel *et* ex-chaînes vides (cf. 2.2)
    .groupBy("publisher")
    .agg(
        F.count("*").alias("nombre_jeux"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
        F.sum("positive").alias("avis_positifs"),
        F.sum("negative").alias("avis_negatifs"),
        F.sum("owners_mid").alias("possesseurs_estimes"),
    )
    .withColumn(
        "ratio_positif_pondere",
        F.when(
            (F.col("avis_positifs") + F.col("avis_negatifs")) > 0,
            F.round(100.0 * F.col("avis_positifs")
                    / (F.col("avis_positifs") + F.col("avis_negatifs")), 2),
        ),
    )
    .orderBy(F.desc("nombre_jeux"))
)

# Visualisation Databricks conseillée : bar chart (publisher / nombre_jeux)
display(df_publishers.limit(20))

nb_editeurs = df_games.select("publisher").na.drop().distinct().count()
print(f"Éditeurs distincts (hors valeurs manquantes) : {nb_editeurs:,}")
print("Réserve : un même champ peut contenir plusieurs éditeurs séparés par une virgule,")
print("et certains noms contiennent eux-mêmes une virgule (ex. 'KOEI TECMO GAMES CO., LTD.').")
print("Ce comptage est donc une borne haute du nombre réel d'entités éditrices.")

publisher,nombre_jeux,prix_moyen_usd,avis_positifs,avis_negatifs,possesseurs_estimes,ratio_positif_pondere
Big Fish Games,422,9.5,5683,1853,5165000.0,75.41
8floor,202,4.99,1125,360,2020000.0,75.76
SEGA,165,14.35,816595,120824,9.2305E7,87.11
Strategy First,150,6.97,53247,14585,7790000.0,78.5
Square Enix,141,20.82,804153,162919,7.757E7,83.15
Choice of Games,140,5.35,6743,1074,1540000.0,86.26
Sekai Project,132,14.16,121117,7121,8470000.0,94.45
HH-Games,132,5.05,1615,676,1445000.0,70.49
Ubisoft,127,17.72,2505190,493686,1.72565E8,83.54
Laush Studio,126,7.11,3276,1289,2695000.0,71.76


Éditeurs distincts (hors valeurs manquantes) : 29,497
Réserve : un même champ peut contenir plusieurs éditeurs séparés par une virgule,
et certains noms contiennent eux-mêmes une virgule (ex. 'KOEI TECMO GAMES CO., LTD.').
Ce comptage est donc une borne haute du nombre réel d'entités éditrices.


## 3.2 Quels sont les jeux les mieux notés ?

Le ratio brut `positifs / total` favorise mécaniquement les jeux confidentiels : un titre
à 104 avis et 100 % de positifs bat Portal 2 et ses 300 000 avis. Deux parades :

- un **seuil minimal d'avis**, mais il reste arbitraire ;
- le **score de Wilson** (borne basse de l'intervalle de confiance à 95 %), qui intègre
  nativement le volume d'avis dans le classement.

Les deux classements sont affichés côte à côte : l'écart est en lui-même un résultat.

In [0]:
SEUIL_AVIS = 500

df_qualite = df_games.filter(F.col("total_reviews") >= SEUIL_AVIS)

print(f"Jeux retenus (≥ {SEUIL_AVIS} avis) : {df_qualite.count():,}")

print("\n▶ Classement A — score de Wilson (recommandé)")
display(
    df_qualite.select("name", "publisher", "positive", "negative",
                      "total_reviews", "ratio_positif", "score_wilson", "price_usd")
              .orderBy(F.desc("score_wilson"))
              .limit(20)
)

Jeux retenus (≥ 500 avis) : 7,691

▶ Classement A — score de Wilson (recommandé)


name,publisher,positive,negative,total_reviews,ratio_positif,score_wilson,price_usd
Flowers -Le volume sur ete-,JAST USA,937,1,938,99.89,99.4,19.99
A Short Hike,adamgryu,11645,87,11732,99.26,99.09,7.99
Senren＊Banka,"HIKARI FIELD, NekoNyan Ltd.",10593,84,10677,99.21,99.03,34.99
Aventura Copilului Albastru și Urât,Codrin Bradea,2203,14,2217,99.37,98.94,1.99
People Playground,Studio Minus,142920,1649,144569,98.86,98.8,9.99
Portal 2,Valve,305671,3770,309441,98.78,98.74,9.99
CULTIC,3D Realms,2021,16,2037,99.21,98.73,9.99
Vampire Survivors,poncle,130311,1624,131935,98.77,98.71,4.99
Patrick's Parabox,Patrick Traynor,1489,11,1500,99.27,98.69,15.99
Ib,PLAYISM,1814,15,1829,99.18,98.65,12.99


In [0]:
print("▶ Classement B — ratio brut, seuil bas (100 avis) : à titre de comparaison")
print("  Ce classement est dominé par des titres confidentiels à 100 % de ratio.")
print("  C'est exactement le biais que le score de Wilson corrige.\n")

display(
    df_games.filter(F.col("total_reviews") >= 100)
            .select("name", "positive", "negative", "total_reviews",
                    "ratio_positif", "score_wilson")
            .orderBy(F.desc("ratio_positif"), F.desc("total_reviews"))
            .limit(20)
)

▶ Classement B — ratio brut, seuil bas (100 avis) : à titre de comparaison
  Ce classement est dominé par des titres confidentiels à 100 % de ratio.
  C'est exactement le biais que le score de Wilson corrige.



name,positive,negative,total_reviews,ratio_positif,score_wilson
The Void Rains Upon Her Heart,496,0,496,100.0,99.23
祈風 Inorikaze,327,0,327,100.0,98.84
秘封旅行 ~ Secret Sealing Travel,218,0,218,100.0,98.27
Elasto Mania Remastered,190,0,190,100.0,98.02
Freshly Frosted,157,0,157,100.0,97.61
HAYAI,148,0,148,100.0,97.47
FIND ALL 2: Middle Ages,132,0,132,100.0,97.17
Touhou Kaeizuka ～ Phantasmagoria of Flower View.,119,0,119,100.0,96.87
Lucy Dreaming,118,0,118,100.0,96.85
未来战士,116,0,116,100.0,96.79


In [0]:
# Ratio moyen NON pondéré vs ratio PONDÉRÉ : deux réponses à deux questions différentes.
stats_avis = df_games.agg(
    F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen_non_pondere"),
    F.sum("positive").alias("total_positifs"),
    F.sum("negative").alias("total_negatifs"),
    F.sum(F.when(F.col("total_reviews") > 0, 1).otherwise(0)).alias("jeux_avec_avis"),
    F.round(F.avg("total_reviews"), 0).alias("avis_moyen_par_jeu"),
    F.expr("percentile_approx(total_reviews, 0.5)").alias("avis_median_par_jeu"),
).collect()[0]

ratio_pondere = round(
    100.0 * stats_avis["total_positifs"]
    / (stats_avis["total_positifs"] + stats_avis["total_negatifs"]), 2
)

print(f"Jeux disposant d'au moins un avis : {stats_avis['jeux_avec_avis']:,} / {NB_JEUX:,}")
print(f"Avis par jeu — moyenne : {stats_avis['avis_moyen_par_jeu']:,.0f} | médiane : {stats_avis['avis_median_par_jeu']:,.0f}")
print(f"\nRatio positif MOYEN (moyenne des ratios par jeu) : {stats_avis['ratio_moyen_non_pondere']} %")
print(f"  -> 'le jeu Steam médian est apprécié à {stats_avis['ratio_moyen_non_pondere']} %'")
print(f"\nRatio positif PONDÉRÉ (somme des avis)          : {ratio_pondere} %")
print(f"  -> 'l'avis Steam moyen est positif à {ratio_pondere} %'")
print("\nL'écart entre les deux vient du poids des blockbusters, très bien notés")
print("et porteurs de la grande majorité des avis. Les deux chiffres sont justes ;")
print("ils ne répondent simplement pas à la même question.")

Jeux disposant d'au moins un avis : 54,566 / 54,725
Avis par jeu — moyenne : 1,737 | médiane : 26

Ratio positif MOYEN (moyenne des ratios par jeu) : 73.74 %
  -> 'le jeu Steam médian est apprécié à 73.74 %'

Ratio positif PONDÉRÉ (somme des avis)          : 85.88 %
  -> 'l'avis Steam moyen est positif à 85.88 %'

L'écart entre les deux vient du poids des blockbusters, très bien notés
et porteurs de la grande majorité des avis. Les deux chiffres sont justes ;
ils ne répondent simplement pas à la même question.


## 3.3 Y a-t-il des années avec plus de sorties ? Quel a été l'effet du Covid ?

In [0]:
# Contrôle du parsing des dates avant toute conclusion.
display(df_games.select("release_date", "release_year", "release_dt", "release_month").limit(10))

couverture = df_games.agg(
    F.sum(F.when(F.col("release_year").isNotNull(), 1).otherwise(0)).alias("annee_ok"),
    F.sum(F.when(F.col("release_dt").isNotNull(), 1).otherwise(0)).alias("date_ok"),
).collect()[0]

print(f"Année extraite      : {couverture['annee_ok']:,} / {NB_JEUX:,}  ({100.0*couverture['annee_ok']/NB_JEUX:.1f} %)")
print(f"Date complète parsée: {couverture['date_ok']:,} / {NB_JEUX:,}  ({100.0*couverture['date_ok']/NB_JEUX:.1f} %)")
if couverture["date_ok"] < 0.5 * couverture["annee_ok"]:
    print("\n⚠️ Faible couverture du parsing de la date complète : l'analyse mensuelle")
    print("   sera partielle. L'analyse annuelle, fondée sur une regex, reste fiable.")

release_date,release_year,release_dt,release_month
2000/11/1,2000,2000-11-01,2000-11
2021/05/14,2021,2021-05-14,2021-05
2020/10/16,2020,2020-10-16,2020-10
2020/10/14,2020,2020-10-14,2020-10
2019/03/30,2019,2019-03-30,2019-03
2019/06/24,2019,2019-06-24,2019-06
2019/01/24,2019,2019-01-24,2019-01
2019/04/8,2019,2019-04-08,2019-04
2019/01/6,2019,2019-01-06,2019-01
2021/09/9,2021,2021-09-09,2021-09


Année extraite      : 54,633 / 54,725  (99.8 %)
Date complète parsée: 54,516 / 54,725  (99.6 %)


In [0]:
# --- Sorties par année --------------------------------------------------------
# Visualisation Databricks conseillée : line chart (release_year / nombre_jeux)

sorties_par_annee = (
    df_games
    .filter(F.col("release_year").between(2004, 2023))   # bornes : lancement de Steam -> extraction
    .groupBy("release_year")
    .agg(
        F.count("*").alias("nombre_jeux"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
        F.round(F.avg("ratio_positif"), 2).alias("ratio_positif_moyen"),
    )
    .orderBy("release_year")
)

display(sorties_par_annee)

release_year,nombre_jeux,prix_moyen_usd,ratio_positif_moyen
2004,6,9.99,90.8
2005,6,7.99,82.68
2006,61,9.02,84.68
2007,98,6.72,80.82
2008,159,8.53,77.63
2009,311,9.22,76.67
2010,288,8.0,76.12
2011,267,9.07,74.2
2012,341,9.36,76.2
2013,458,10.05,74.86


In [0]:
# --- Zoom Covid : rythme mensuel des sorties 2018 -> 2022 ---------------------
# Visualisation Databricks conseillée : line chart (release_month / nombre_jeux)

sorties_mensuelles = (
    df_games
    .filter(F.col("release_month").isNotNull())
    .filter(F.col("release_year").between(2018, 2022))
    .groupBy("release_month")
    .agg(F.count("*").alias("nombre_jeux"))
    .orderBy("release_month")
)

display(sorties_mensuelles)

release_month,nombre_jeux
2018-01,491
2018-02,609
2018-03,733
2018-04,651
2018-05,671
2018-06,580
2018-07,631
2018-08,641
2018-09,641
2018-10,602


In [0]:
# --- Lecture chiffrée de l'effet Covid ----------------------------------------
annees = {r["release_year"]: r["nombre_jeux"]
          for r in sorties_par_annee.collect()}

def variation(a, b):
    if annees.get(a) and annees.get(b):
        return f"{100.0 * (annees[b] - annees[a]) / annees[a]:+.1f} %"
    return "n/d"

print("Sorties de jeux par année (période Covid) :")
for an in [2017, 2018, 2019, 2020, 2021, 2022]:
    if an in annees:
        print(f"  {an} : {annees[an]:>6,} jeux")

print("\nVariations annuelles :")
print(f"  2018 -> 2019 (avant Covid) : {variation(2018, 2019)}")
print(f"  2019 -> 2020 (année Covid) : {variation(2019, 2020)}")
print(f"  2020 -> 2021 (post-Covid)  : {variation(2020, 2021)}")
print("\nRéserve de lecture : le dataset est une photographie à un instant t. Les jeux")
print("retirés du catalogue depuis leur sortie sont absents, ce qui sous-estime")
print("mécaniquement les années anciennes. Les dernières années peuvent également")
print("être tronquées si l'extraction a eu lieu en cours d'année.")

Sorties de jeux par année (période Covid) :
  2017 :  5,845 jeux
  2018 :  7,510 jeux
  2019 :  6,815 jeux
  2020 :  8,208 jeux
  2021 :  8,722 jeux
  2022 :  7,411 jeux

Variations annuelles :
  2018 -> 2019 (avant Covid) : -9.3 %
  2019 -> 2020 (année Covid) : +20.4 %
  2020 -> 2021 (post-Covid)  : +6.3 %

Réserve de lecture : le dataset est une photographie à un instant t. Les jeux
retirés du catalogue depuis leur sortie sont absents, ce qui sous-estime
mécaniquement les années anciennes. Les dernières années peuvent également
être tronquées si l'extraction a eu lieu en cours d'année.


## 3.4 Comment les prix sont-ils distribués ?

⚠️ Rappel : `price` est en **centimes de dollar** et stocké en **texte**. Toutes les valeurs
ci-dessous utilisent `price_usd`, la version castée et convertie (cf. 2.3).

In [0]:
stats_prix = df_games.agg(
    F.sum(F.when(F.col("price_usd").isNotNull(), 1).otherwise(0)).alias("jeux_avec_prix"),
    F.round(F.avg("price_usd"), 2).alias("prix_moyen"),
    F.expr("percentile_approx(price_usd, 0.5)").alias("prix_median"),
    F.round(F.min("price_usd"), 2).alias("prix_min"),
    F.round(F.max("price_usd"), 2).alias("prix_max"),
    F.sum(F.when(F.col("price_usd") == 0, 1).otherwise(0)).alias("jeux_gratuits"),
    F.expr("percentile_approx(price_usd, 0.9)").alias("prix_p90"),
).collect()[0]

print("Statistiques de prix (en dollars US)")
print("-" * 46)
print(f"Jeux avec un prix renseigné : {stats_prix['jeux_avec_prix']:,}")
print(f"Prix moyen                  : {stats_prix['prix_moyen']:>8.2f} $")
print(f"Prix médian                 : {stats_prix['prix_median']:>8.2f} $")
print(f"9e décile (P90)             : {stats_prix['prix_p90']:>8.2f} $")
print(f"Prix minimum                : {stats_prix['prix_min']:>8.2f} $")
print(f"Prix maximum                : {stats_prix['prix_max']:>8.2f} $")
print(f"Jeux gratuits (0 $)         : {stats_prix['jeux_gratuits']:,} "
      f"({100.0 * stats_prix['jeux_gratuits'] / NB_JEUX:.1f} %)")
print("\nLe prix moyen est tiré vers le haut par une minorité de titres premium :")
print("la médiane est l'indicateur à retenir pour décrire le marché.")

Statistiques de prix (en dollars US)
----------------------------------------------
Jeux avec un prix renseigné : 54,725
Prix moyen                  :     7.58 $
Prix médian                 :     4.99 $
9e décile (P90)             :    19.99 $
Prix minimum                :     0.00 $
Prix maximum                :   999.00 $
Jeux gratuits (0 $)         : 7,411 (13.5 %)

Le prix moyen est tiré vers le haut par une minorité de titres premium :
la médiane est l'indicateur à retenir pour décrire le marché.


In [0]:
# --- Répartition par tranche de prix ------------------------------------------
# Visualisation Databricks conseillée : bar chart (tranche_prix / nombre_jeux)

ORDRE_TRANCHES = ["Gratuit", "0-5 $", "5-10 $", "10-20 $", "20-40 $", "40-60 $", "60 $ et +", "Non renseigné"]

distribution_prix = (
    df_games
    .withColumn(
        "tranche_prix",
        F.when(F.col("price_usd").isNull(), "Non renseigné")
         .when(F.col("price_usd") == 0, "Gratuit")
         .when(F.col("price_usd") <= 5, "0-5 $")
         .when(F.col("price_usd") <= 10, "5-10 $")
         .when(F.col("price_usd") <= 20, "10-20 $")
         .when(F.col("price_usd") <= 40, "20-40 $")
         .when(F.col("price_usd") <= 60, "40-60 $")
         .otherwise("60 $ et +"),
    )
    .groupBy("tranche_prix")
    .agg(F.count("*").alias("nombre_jeux"))
    .withColumn("pct_catalogue", F.round(100.0 * F.col("nombre_jeux") / NB_JEUX, 2))
    .withColumn("ordre", F.array_position(F.array(*[F.lit(x) for x in ORDRE_TRANCHES]),
                                          F.col("tranche_prix")))
    .orderBy("ordre")
    .drop("ordre")
)

display(distribution_prix)

tranche_prix,nombre_jeux,pct_catalogue
Gratuit,7411,13.54
0-5 $,23330,42.63
5-10 $,12340,22.55
10-20 $,8890,16.24
20-40 $,2296,4.2
40-60 $,395,0.72
60 $ et +,63,0.12


## 3.5 Y a-t-il beaucoup de jeux en promotion ?

In [0]:
promo = (
    df_games
    .withColumn(
        "statut_promo",
        F.when(F.col("discount_pct").isNull(), "Non renseigné")
         .when(F.col("discount_pct") > 0, "En promotion")
         .otherwise("Prix plein"),
    )
    .groupBy("statut_promo")
    .agg(F.count("*").alias("nombre_jeux"))
    .withColumn("pct_catalogue", F.round(100.0 * F.col("nombre_jeux") / NB_JEUX, 2))
    .orderBy(F.desc("nombre_jeux"))
)

display(promo)

statut_promo,nombre_jeux,pct_catalogue
Prix plein,52230,95.44
En promotion,2495,4.56


In [0]:
stats_promo = df_games.filter(F.col("discount_pct") > 0).agg(
    F.count("*").alias("jeux_en_promo"),
    F.round(F.avg("discount_pct"), 2).alias("remise_moyenne_pct"),
    F.expr("percentile_approx(discount_pct, 0.5)").alias("remise_mediane_pct"),
    F.round(F.min("discount_pct"), 2).alias("remise_min_pct"),
    F.round(F.max("discount_pct"), 2).alias("remise_max_pct"),
    F.round(F.avg("initialprice_usd"), 2).alias("prix_initial_moyen"),
    F.round(F.avg("price_usd"), 2).alias("prix_remise_moyen"),
).collect()[0]

print("Jeux actuellement en promotion")
print("-" * 46)
print(f"Nombre                : {stats_promo['jeux_en_promo']:,} "
      f"({100.0 * stats_promo['jeux_en_promo'] / NB_JEUX:.1f} % du catalogue)")
print(f"Remise moyenne        : {stats_promo['remise_moyenne_pct']:.1f} %")
print(f"Remise médiane        : {stats_promo['remise_mediane_pct']:.0f} %")
print(f"Remise min / max      : {stats_promo['remise_min_pct']:.0f} % / {stats_promo['remise_max_pct']:.0f} %")
print(f"Prix initial moyen    : {stats_promo['prix_initial_moyen']:.2f} $")
print(f"Prix remisé moyen     : {stats_promo['prix_remise_moyen']:.2f} $")
print("\nRéserve : ce chiffre est une photographie instantanée. Il mesure les jeux")
print("en promotion AU MOMENT DE L'EXTRACTION, pas la part des jeux qui ont connu")
print("au moins une promotion dans leur vie — cette dernière est bien plus élevée.")

Jeux actuellement en promotion
----------------------------------------------
Nombre                : 2,495 (4.6 % du catalogue)
Remise moyenne        : 57.6 %
Remise médiane        : 60 %
Remise min / max      : 10 % / 90 %
Prix initial moyen    : 9.28 $
Prix remisé moyen     : 4.03 $

Réserve : ce chiffre est une photographie instantanée. Il mesure les jeux
en promotion AU MOMENT DE L'EXTRACTION, pas la part des jeux qui ont connu
au moins une promotion dans leur vie — cette dernière est bien plus élevée.


In [0]:
# --- Distribution des taux de remise -------------------------------------------
# Visualisation Databricks conseillée : bar chart (tranche_remise / nombre_jeux)

display(
    df_games.filter(F.col("discount_pct") > 0)
    .withColumn(
        "tranche_remise",
        F.when(F.col("discount_pct") < 25, "1-24 %")
         .when(F.col("discount_pct") < 50, "25-49 %")
         .when(F.col("discount_pct") < 75, "50-74 %")
         .otherwise("75 % et +"),
    )
    .groupBy("tranche_remise")
    .agg(F.count("*").alias("nombre_jeux"))
    .orderBy("tranche_remise")
)

tranche_remise,nombre_jeux
1-24 %,223
25-49 %,501
50-74 %,929
75 % et +,842


## 3.6 Quelles sont les langues les plus représentées ?

Le champ `languages` de SteamSpy contient des balises HTML, des astérisques et une mention
de fin (`languages with full audio support`). Il est nettoyé en amont (cf. 2.3), puis éclaté
et **dédoublonné par jeu** : un jeu qui déclare deux fois l'anglais ne doit être compté qu'une fois.

*Périmètre : `df_clean` — l'offre linguistique concerne tout le catalogue.*

In [0]:
df_langues = (
    df_clean
    .filter(F.col("languages_clean").isNotNull() & (F.col("languages_clean") != ""))
    .select("appid", F.explode(F.split(F.col("languages_clean"), ",")).alias("langue"))
    .withColumn("langue", F.trim(F.col("langue")))
    .filter(F.col("langue") != "")
    .dropDuplicates(["appid", "langue"])
)

# Visualisation Databricks conseillée : bar chart (langue / nombre_jeux)
top_langues = (
    df_langues.groupBy("langue")
    .agg(F.countDistinct("appid").alias("nombre_jeux"))
    .withColumn("pct_catalogue", F.round(100.0 * F.col("nombre_jeux") / NB_LIGNES, 2))
    .orderBy(F.desc("nombre_jeux"))
    .limit(20)
)

display(top_langues)

langue,nombre_jeux,pct_catalogue
English,55116,98.97
German,14019,25.17
French,13426,24.11
Russian,12922,23.2
Simplified Chinese,12782,22.95
Spanish - Spain,12233,21.97
Japanese,10368,18.62
Italian,9304,16.71
Portuguese - Brazil,6750,12.12
Korean,6600,11.85


In [0]:
stats_langues = df_clean.agg(
    F.round(F.avg("nb_langues"), 2).alias("moyenne"),
    F.expr("percentile_approx(nb_langues, 0.5)").alias("mediane"),
    F.max("nb_langues").alias("maximum"),
    F.sum(F.when(F.col("nb_langues") == 1, 1).otherwise(0)).alias("monolingues"),
).collect()[0]

print("Nombre de langues supportées par jeu")
print("-" * 46)
print(f"Moyenne  : {stats_langues['moyenne']}")
print(f"Médiane  : {stats_langues['mediane']:.0f}")
print(f"Maximum  : {stats_langues['maximum']}")
print(f"Jeux monolingues : {stats_langues['monolingues']:,} "
      f"({100.0 * stats_langues['monolingues'] / NB_LIGNES:.1f} % du catalogue)")
print("\nLe lien entre nombre de langues et succès est testé en partie 6.")

Nombre de langues supportées par jeu
----------------------------------------------
Moyenne  : 3.63
Médiane  : 1
Maximum  : 40
Jeux monolingues : 29,655 (53.2 % du catalogue)

Le lien entre nombre de langues et succès est testé en partie 6.


## 3.7 Y a-t-il beaucoup de jeux interdits aux moins de 16 / 18 ans ?

La classification par âge se lit dans le champ **`required_age`**, et non dans `categories`
(qui ne contient que des fonctionnalités Steam : *Single-player*, *Steam Cloud*…).
Elle est complétée par les **genres de contenu sensible** que Steam expose
(`Violent`, `Gore`, `Sexual Content`, `Nudity`), qui constituent un second signal.

In [0]:
# Valeurs brutes rencontrées, avant tout cast — pour vérifier qu'aucun format n'est perdu.
display(
    df_clean.groupBy("required_age")
            .agg(F.count("*").alias("nombre"))
            .orderBy(F.desc("nombre"))
            .limit(20)
)

required_age,nombre
0,55030
15,264
18,223
17,38
16,38
12,32
13,26
14,10
10,7
6,4


In [0]:
# Visualisation Databricks conseillée : bar chart (classe_age / nombre_jeux)

classement_age = (
    df_games
    .withColumn(
        "classe_age",
        F.when(F.col("required_age_int").isNull(), "Non renseigné")
         .when(F.col("required_age_int") == 0, "Tout public (0)")
         .when(F.col("required_age_int") < 16, "Moins de 16")
         .when(F.col("required_age_int") < 18, "16-17")
         .otherwise("18 et +"),
    )
    .groupBy("classe_age")
    .agg(F.count("*").alias("nombre_jeux"))
    .withColumn("pct_jeux", F.round(100.0 * F.col("nombre_jeux") / NB_JEUX, 2))
    .orderBy(F.desc("nombre_jeux"))
)

display(classement_age)

classe_age,nombre_jeux,pct_jeux
Tout public (0),54064,98.79
Moins de 16,355,0.65
18 et +,230,0.42
16-17,76,0.14


In [0]:
nb_18   = df_games.filter(F.col("required_age_int") >= 18).count()
nb_1617 = df_games.filter(F.col("required_age_int").between(16, 17)).count()
nb_age_renseigne = df_games.filter(F.col("required_age_int").isNotNull()).count()
nb_age_positif   = df_games.filter(F.col("required_age_int") > 0).count()

print("Réponse à la question du cahier des charges")
print("-" * 46)
print(f"Jeux interdits aux moins de 18 ans : {nb_18:,}  ({100.0*nb_18/NB_JEUX:.2f} % du catalogue)")
print(f"Jeux classés 16-17 ans             : {nb_1617:,}  ({100.0*nb_1617/NB_JEUX:.2f} %)")
print(f"Jeux avec une restriction d'âge > 0 : {nb_age_positif:,}  ({100.0*nb_age_positif/NB_JEUX:.2f} %)")
print(f"\nLe champ `required_age` est TOUJOURS présent ({100.0*nb_age_renseigne/NB_JEUX:.1f} % des jeux),")
print(f"mais il vaut 0 pour {100.0*(NB_JEUX-nb_age_positif)/NB_JEUX:.1f} % d'entre eux.")
print("Réserve importante : Steam n'impose une classification que pour certains")
print("contenus, et la déclaration reste à la main de l'éditeur. Une valeur 0 ne")
print("signifie donc pas 'tout public vérifié' mais 'aucune restriction déclarée'.")
print("Les chiffres ci-dessus sont des BORNES BASSES.")

In [0]:
# --- Second signal : les genres de contenu sensible ----------------------------
GENRES_SENSIBLES = ["Violent", "Gore", "Sexual Content", "Nudity"]

df_sensible = (
    df_games
    .filter(F.col("genre").isNotNull())
    .select("appid", "name",
            F.explode(F.split(F.col("genre"), ",")).alias("g"))
    .withColumn("g", F.trim(F.col("g")))
    .filter(F.col("g").isin(GENRES_SENSIBLES))
)

display(
    df_sensible.groupBy("g")
               .agg(F.countDistinct("appid").alias("nombre_jeux"))
               .orderBy(F.desc("nombre_jeux"))
)

nb_sensible = df_sensible.select("appid").distinct().count()
print(f"Jeux portant au moins un genre de contenu sensible : {nb_sensible:,} "
      f"({100.0 * nb_sensible / NB_JEUX:.2f} % du catalogue)")

g,nombre_jeux
Violent,168
Gore,99
Sexual Content,54
Nudity,45


Jeux portant au moins un genre de contenu sensible : 222 (0.41 % du catalogue)


## 3.8 Quelles fonctionnalités Steam sont les plus répandues ?

*Périmètre : `df_clean`.* Attention : `explode` écarte silencieusement les entrées
sans catégorie. Le dénominateur utilisé pour les pourcentages est donc le nombre
d'entrées **qui déclarent au moins une catégorie**, pas le catalogue entier.

In [0]:
nb_avec_categories = df_clean.filter(F.col("nb_categories") > 0).count()

print(f"Entrées déclarant au moins une fonctionnalité : {nb_avec_categories:,} / {NB_LIGNES:,} "
      f"({100.0 * nb_avec_categories / NB_LIGNES:.1f} %)")
print(f"Entrées sans aucune fonctionnalité déclarée   : {NB_LIGNES - nb_avec_categories:,}")
print("Les pourcentages ci-dessous sont calculés sur la première population.")

df_categories = (
    df_clean
    .filter(F.col("categories").isNotNull())
    .select("appid", F.explode(F.col("categories")).alias("categorie"))
)

# Visualisation Databricks conseillée : bar chart (categorie / nombre_jeux)
display(
    df_categories.groupBy("categorie")
    .agg(F.countDistinct("appid").alias("nombre_jeux"))
    .withColumn("pct", F.round(100.0 * F.col("nombre_jeux") / nb_avec_categories, 2))
    .orderBy(F.desc("nombre_jeux"))
    .limit(20)
)

Entrées déclarant au moins une fonctionnalité : 54,721 / 55,691 (98.3 %)
Entrées sans aucune fonctionnalité déclarée   : 970
Les pourcentages ci-dessous sont calculés sur la première population.


categorie,nombre_jeux,pct
Single-player,52025,95.07
Steam Achievements,27394,50.06
Steam Cloud,14235,26.01
Full controller support,11879,21.71
Multi-player,11455,20.93
Steam Trading Cards,9208,16.83
Partial Controller Support,7867,14.38
PvP,7070,12.92
Co-op,5616,10.26
Steam Leaderboards,5509,10.07


## 3.9 Synthèse macro

In [0]:
synthese_macro = df_games.agg(
    F.count("*").alias("nb_jeux"),
    F.round(F.avg("price_usd"), 2).alias("prix_moyen"),
    F.expr("percentile_approx(price_usd, 0.5)").alias("prix_median"),
    F.sum(F.when(F.col("price_usd") == 0, 1).otherwise(0)).alias("nb_gratuits"),
    F.sum(F.when(F.col("discount_pct") > 0, 1).otherwise(0)).alias("nb_promo"),
    F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen"),
    F.sum("owners_mid").alias("possesseurs_total"),
    F.round(F.avg("nb_langues"), 1).alias("langues_moy"),
    F.round(F.avg("nb_plateformes"), 2).alias("plateformes_moy"),
).collect()[0]

print("=" * 62)
print(" SYNTHÈSE MACRO — marché du jeu vidéo sur Steam")
print("=" * 62)
print(f" Jeux analysés                 : {synthese_macro['nb_jeux']:,}")
print(f" Éditeurs distincts            : {nb_editeurs:,}")
print(f" Prix moyen / médian           : {synthese_macro['prix_moyen']:.2f} $ / {synthese_macro['prix_median']:.2f} $")
print(f" Jeux gratuits                 : {synthese_macro['nb_gratuits']:,} "
      f"({100.0*synthese_macro['nb_gratuits']/synthese_macro['nb_jeux']:.1f} %)")
print(f" Jeux en promotion (instantané): {synthese_macro['nb_promo']:,} "
      f"({100.0*synthese_macro['nb_promo']/synthese_macro['nb_jeux']:.1f} %)")
print(f" Ratio positif moyen           : {synthese_macro['ratio_moyen']:.2f} %")
print(f" Ratio positif pondéré         : {ratio_pondere:.2f} %")
print(f" Possesseurs cumulés estimés   : {synthese_macro['possesseurs_total']/1e9:.2f} milliards")
print(f" Langues par jeu (moyenne)     : {synthese_macro['langues_moy']}")
print(f" Plateformes par jeu (moyenne) : {synthese_macro['plateformes_moy']}")
print("=" * 62)

 SYNTHÈSE MACRO — marché du jeu vidéo sur Steam
 Jeux analysés                 : 54,725
 Éditeurs distincts            : 29,497
 Prix moyen / médian           : 7.58 $ / 4.99 $
 Jeux gratuits                 : 7,411 (13.5 %)
 Jeux en promotion (instantané): 2,495 (4.6 %)
 Ratio positif moyen           : 73.74 %
 Ratio positif pondéré         : 85.88 %
 Possesseurs cumulés estimés   : 7.39 milliards
 Langues par jeu (moyenne)     : 3.6
 Plateformes par jeu (moyenne) : 1.38


---
# 4. Analyse par genre

*Périmètre : `df_games`.*

⚠️ **Double comptage assumé.** Un jeu déclare en moyenne 2 à 4 genres. Après `explode`,
la somme des jeux par genre dépasse largement le nombre de jeux du catalogue. Les colonnes
sont donc nommées explicitement, et `countDistinct("appid")` est utilisé partout où il faut
compter des jeux plutôt que des occurrences.

In [0]:
# --- 4.1 Construction UNIQUE du dataframe des genres --------------------------
# Deux filtres appliqués ICI, une seule fois : genres vides, et genres non ludiques
# (cf. 2.6). Toutes les cellules en aval héritent d'un dataframe déjà propre.

df_genres = (
    df_games
    .filter(F.col("genre").isNotNull())
    .select(
        "appid", "name", "publisher", "price_usd", "initialprice_usd", "discount_pct",
        "positive", "negative", "total_reviews", "ratio_positif", "score_wilson",
        "owners_mid", "ccu", "release_year", "nb_langues",
        "windows", "mac", "linux",
        F.explode(F.split(F.col("genre"), ",")).alias("genre_simple"),
    )
    .withColumn("genre_simple", F.trim(F.col("genre_simple")))
    .filter((F.col("genre_simple").isNotNull()) & (F.col("genre_simple") != ""))
    .filter(~F.col("genre_simple").isin(GENRES_LOGICIELS))
)

nb_occurrences = df_genres.count()
nb_jeux_avec_genre = df_genres.select("appid").distinct().count()

print(f"Jeux avec au moins un genre : {nb_jeux_avec_genre:,} / {NB_JEUX:,}")
print(f"Occurrences (jeu × genre)   : {nb_occurrences:,}")
print(f"Genres par jeu (moyenne)    : {nb_occurrences / nb_jeux_avec_genre:.2f}")
print(f"\nLes {len(GENRES_LOGICIELS)} catégories logicielles de Steam sont exclues des")
print("classements de genres : ce ne sont pas des genres de jeu. Sans ce filtre,")
print("elles occupent les dix premières places du classement des prix et la")
print("première place du classement de qualité.")

Jeux avec au moins un genre : 54,565 / 54,725
Occurrences (jeu × genre)   : 154,407
Genres par jeu (moyenne)    : 2.83

Les 12 catégories logicielles de Steam sont exclues des
classements de genres : ce ne sont pas des genres de jeu. Sans ce filtre,
elles occupent les dix premières places du classement des prix et la
première place du classement de qualité.


In [0]:
# --- 4.2 Genres les plus représentés ------------------------------------------
# Visualisation Databricks conseillée : bar chart (genre_simple / nombre_jeux)

top_genres = (
    df_genres.groupBy("genre_simple")
    .agg(
        F.countDistinct("appid").alias("nombre_jeux"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
        F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen"),
    )
    .withColumn("pct_des_jeux", F.round(100.0 * F.col("nombre_jeux") / nb_jeux_avec_genre, 2))
    .orderBy(F.desc("nombre_jeux"))
    .limit(20)
)

display(top_genres)

print("Lecture : `pct_des_jeux` indique la part des jeux qui portent CE genre.")
print("La somme des pourcentages dépasse 100 % — c'est normal, un jeu a plusieurs genres.")

genre_simple,nombre_jeux,prix_moyen_usd,ratio_moyen,pct_des_jeux
Indie,39681,6.57,74.21,72.72
Action,23759,7.73,73.0,43.54
Casual,22086,5.61,74.34,40.48
Adventure,21431,8.01,73.88,39.28
Strategy,10895,8.4,71.93,19.97
Simulation,10836,9.09,69.37,19.86
RPG,9534,9.04,73.06,17.47
Early Access,6145,8.75,70.66,11.26
Free to Play,3393,0.29,72.19,6.22
Sports,2666,8.95,69.8,4.89


Lecture : `pct_des_jeux` indique la part des jeux qui portent CE genre.
La somme des pourcentages dépasse 100 % — c'est normal, un jeu a plusieurs genres.


In [0]:
# --- 4.3 Quels genres ont le meilleur ratio positif / négatif ? ---------------
# Les deux métriques sont affichées côte à côte : la moyenne des ratios (chaque
# jeu compte pour 1) et le ratio pondéré (chaque avis compte pour 1).
# Visualisation Databricks conseillée : bar chart (genre_simple / ratio_pondere)

SEUIL_JEUX_GENRE = 50

qualite_genres = (
    df_genres
    .filter(F.col("total_reviews") > 0)
    .groupBy("genre_simple")
    .agg(
        F.countDistinct("appid").alias("nombre_jeux"),
        F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen_non_pondere"),
        F.sum("positive").alias("avis_positifs"),
        F.sum("negative").alias("avis_negatifs"),
        F.round(F.avg("score_wilson"), 2).alias("wilson_moyen"),
        F.round(F.avg("total_reviews"), 0).alias("avis_moyen_par_jeu"),
    )
    .filter(F.col("nombre_jeux") >= SEUIL_JEUX_GENRE)
    .withColumn(
        "ratio_pondere",
        F.round(100.0 * F.col("avis_positifs")
                / (F.col("avis_positifs") + F.col("avis_negatifs")), 2),
    )
    .select("genre_simple", "nombre_jeux", "ratio_moyen_non_pondere",
            "ratio_pondere", "wilson_moyen", "avis_moyen_par_jeu")
    .orderBy(F.desc("ratio_pondere"))
)

display(qualite_genres)

print("Le classement par ratio PONDÉRÉ écarte les genres qui doivent leur bonne note")
print("à une multitude de petits titres. C'est celui à retenir pour un arbitrage éditorial.")

genre_simple,nombre_jeux,ratio_moyen_non_pondere,ratio_pondere,wilson_moyen,avis_moyen_par_jeu
Indie,39558,74.21,88.47,53.19,930.0
Casual,21990,74.34,86.72,51.21,526.0
Simulation,10809,69.37,86.64,51.93,1663.0
Racing,2146,70.22,85.91,50.48,1269.0
RPG,9520,73.06,85.58,55.4,2384.0
Action,23695,73.0,84.99,52.3,2724.0
Strategy,10876,71.93,84.85,52.84,1452.0
Adventure,21397,73.88,84.0,54.4,1652.0
Early Access,6128,70.66,82.24,47.7,860.0
Gore,99,63.2,81.6,46.64,354.0


Le classement par ratio PONDÉRÉ écarte les genres qui doivent leur bonne note
à une multitude de petits titres. C'est celui à retenir pour un arbitrage éditorial.


In [0]:
# --- 4.4 Prix par genre --------------------------------------------------------
# Visualisation Databricks conseillée : bar chart (genre_simple / prix_median_usd)

SEUIL_JEUX_PRIX = 30

prix_par_genre = (
    df_genres
    .filter(F.col("price_usd") > 0)          # on écarte le free-to-play, qui écraserait les moyennes
    .groupBy("genre_simple")
    .agg(
        F.countDistinct("appid").alias("nombre_jeux"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
        F.expr("percentile_approx(price_usd, 0.5)").alias("prix_median_usd"),
        F.round(F.min("price_usd"), 2).alias("prix_min_usd"),
        F.round(F.max("price_usd"), 2).alias("prix_max_usd"),
    )
    .filter(F.col("nombre_jeux") >= SEUIL_JEUX_PRIX)
    .orderBy(F.desc("prix_median_usd"))
)

display(prix_par_genre)

print("Les prix sont exprimés en dollars US (colonne price_usd, castée et divisée par 100).")
print("Périmètre : df_genres — les catégories logicielles de Steam (Audio Production,")
print("Video Production, Utilities...) sont exclues, non par le champ `type` qui est")
print("inopérant sur ce dataset, mais par la liste GENRES_LOGICIELS définie en 2.6.")

genre_simple,nombre_jeux,prix_moyen_usd,prix_median_usd,prix_min_usd,prix_max_usd
Early Access,5046,10.65,9.99,0.37,199.99
Simulation,9451,10.42,7.99,0.28,199.99
Massively Multiplayer,686,10.64,7.99,0.37,59.99
RPG,8140,10.59,7.99,0.29,199.99
Sports,2253,10.59,7.99,0.37,199.99
Adventure,19019,9.02,6.99,0.28,199.99
Strategy,9386,9.76,6.99,0.28,99.99
Action,20582,8.92,5.99,0.28,999.0
Racing,1882,9.41,5.99,0.31,199.99
Indie,34765,7.5,4.99,0.28,199.99


Les prix sont exprimés en dollars US (colonne price_usd, castée et divisée par 100).
Périmètre : df_genres — les catégories logicielles de Steam (Audio Production,
Video Production, Utilities...) sont exclues, non par le champ `type` qui est
inopérant sur ce dataset, mais par la liste GENRES_LOGICIELS définie en 2.6.


In [0]:
# --- 4.5 Quels sont les genres les plus lucratifs ? ---------------------------
# Estimation = prix courant × milieu de la fourchette de possesseurs (`owners`).
# On n'utilise PAS le nombre d'avis comme proxy de ventes : le rapport ventes/avis
# est de l'ordre de 30 à 50× et varie fortement d'un jeu à l'autre.
# Visualisation Databricks conseillée : bar chart (genre_simple / revenu_total_musd)

revenu_par_genre = (
    df_genres
    .filter((F.col("price_usd") > 0) & (F.col("owners_mid") > 0))
    .withColumn("revenu_estime_usd", F.col("price_usd") * F.col("owners_mid"))
    .groupBy("genre_simple")
    .agg(
        F.countDistinct("appid").alias("nombre_jeux"),
        F.round(F.sum("revenu_estime_usd") / 1e6, 1).alias("revenu_total_musd"),
        F.round(F.avg("revenu_estime_usd") / 1e6, 3).alias("revenu_moyen_par_jeu_musd"),
        F.round(F.expr("percentile_approx(revenu_estime_usd, 0.5)"), 0).alias("revenu_median_par_jeu_usd"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
    )
    .filter(F.col("nombre_jeux") >= SEUIL_JEUX_PRIX)
    .orderBy(F.desc("revenu_total_musd"))
)

display(revenu_par_genre)

print("⚠️ TROIS RÉSERVES MÉTHODOLOGIQUES, à énoncer avant toute conclusion :")
print("  1. `owners` est une FOURCHETTE très large (ex. 1M .. 2M) : on en prend le milieu.")
print("  2. Le prix retenu est le prix ACTUEL, pas le prix moyen de vente historique.")
print("     Les jeux anciens, souvent bradés, sont donc sous-évalués.")
print("  3. Un jeu à 4 genres compte son revenu 4 FOIS : la somme des revenus par")
print("     genre dépasse le revenu total du marché. Comparer les genres entre eux,")
print("     jamais additionner les colonnes.")
print("La commission Steam (~30 %) n'est pas déduite : ce sont des revenus bruts.")

genre_simple,nombre_jeux,revenu_total_musd,revenu_moyen_par_jeu_musd,revenu_median_par_jeu_usd,prix_moyen_usd
Action,20582,58756.5,2.855,89900.0,8.92
Adventure,19019,37245.7,1.958,99900.0,9.02
Indie,34765,32346.6,0.93,69900.0,7.5
RPG,8140,27173.1,3.338,109900.0,10.59
Strategy,9386,20150.0,2.147,99900.0,9.76
Simulation,9451,18769.7,1.986,99900.0,10.42
Casual,19418,8081.0,0.416,49900.0,6.38
Massively Multiplayer,686,5930.2,8.645,109900.0,10.64
Early Access,5046,5458.7,1.082,99900.0,10.65
Sports,2253,3149.9,1.398,99900.0,10.59


⚠️ TROIS RÉSERVES MÉTHODOLOGIQUES, à énoncer avant toute conclusion :
  1. `owners` est une FOURCHETTE très large (ex. 1M .. 2M) : on en prend le milieu.
  2. Le prix retenu est le prix ACTUEL, pas le prix moyen de vente historique.
     Les jeux anciens, souvent bradés, sont donc sous-évalués.
  3. Un jeu à 4 genres compte son revenu 4 FOIS : la somme des revenus par
     genre dépasse le revenu total du marché. Comparer les genres entre eux,
     jamais additionner les colonnes.
La commission Steam (~30 %) n'est pas déduite : ce sont des revenus bruts.


In [0]:
# --- 4.6 Les éditeurs ont-ils des genres de prédilection ? --------------------
# Sortie pivotée : un éditeur par ligne, un genre par colonne -> exploitable
# directement en histogramme empilé dans Databricks.

top_10_editeurs = [r["publisher"] for r in
                   df_publishers.select("publisher").limit(10).collect()]

genres_majeurs = [r["genre_simple"] for r in
                  top_genres.select("genre_simple").limit(10).collect()]

print("Top 10 éditeurs analysés :", top_10_editeurs)

specialites = (
    df_genres
    .filter(F.col("publisher").isin(top_10_editeurs))
    .groupBy("publisher")
    .pivot("genre_simple", genres_majeurs)
    .agg(F.countDistinct("appid"))
    .na.fill(0)
)

display(specialites)

Top 10 éditeurs analysés : ['Big Fish Games', '8floor', 'SEGA', 'Strategy First', 'Square Enix', 'Choice of Games', 'HH-Games', 'Sekai Project', 'Ubisoft', 'Laush Studio']


publisher,Indie,Action,Casual,Adventure,Strategy,Simulation,RPG,Early Access,Free to Play,Sports
HH-Games,69,38,132,39,48,12,1,0,0,1
Big Fish Games,7,1,418,392,6,7,0,0,0,2
Strategy First,33,28,25,42,52,25,5,0,0,2
Laush Studio,124,20,87,14,3,18,0,0,0,4
Ubisoft,7,70,11,45,22,18,19,0,6,5
Square Enix,5,75,7,43,13,7,71,0,4,1
Sekai Project,88,21,99,51,8,24,16,5,6,1
8floor,1,0,202,8,22,10,0,0,0,0
Choice of Games,136,13,28,112,0,0,139,0,0,0
SEGA,12,80,13,33,32,17,25,0,2,17


In [0]:
# Même information en format long : plus lisible pour un graphique groupé,
# et met en évidence le degré de spécialisation de chaque éditeur.

specialites_long = (
    df_genres
    .filter(F.col("publisher").isin(top_10_editeurs))
    .groupBy("publisher", "genre_simple")
    .agg(F.countDistinct("appid").alias("nombre_jeux"))
)

total_par_editeur = (
    specialites_long.groupBy("publisher")
    .agg(F.sum("nombre_jeux").alias("total_occurrences"))
)

display(
    specialites_long.join(total_par_editeur, "publisher")
    .withColumn("pct_du_catalogue_editeur",
                F.round(100.0 * F.col("nombre_jeux") / F.col("total_occurrences"), 1))
    .filter(F.col("nombre_jeux") >= 5)
    .orderBy("publisher", F.desc("nombre_jeux"))
)

publisher,genre_simple,nombre_jeux,total_occurrences,pct_du_catalogue_editeur
8floor,Casual,202,243,83.1
8floor,Strategy,22,243,9.1
8floor,Simulation,10,243,4.1
8floor,Adventure,8,243,3.3
Big Fish Games,Casual,418,833,50.2
Big Fish Games,Adventure,392,833,47.1
Big Fish Games,Indie,7,833,0.8
Big Fish Games,Simulation,7,833,0.8
Big Fish Games,Strategy,6,833,0.7
Choice of Games,RPG,139,428,32.5


---
# 5. Analyse par plateforme

*Périmètre : `df_games`.* Les booléens `windows` / `mac` / `linux` ont été aplatis
une fois pour toutes en partie 2 : aucune cellule ne réextrait `data.platforms`.

In [0]:
# --- 5.1 Répartition Windows / Mac / Linux ------------------------------------
# Visualisation Databricks conseillée : bar chart (plateforme / nombre_jeux)

compte_plateformes = df_games.agg(
    F.sum(F.when(F.col("windows"), 1).otherwise(0)).alias("windows"),
    F.sum(F.when(F.col("mac"), 1).otherwise(0)).alias("mac"),
    F.sum(F.when(F.col("linux"), 1).otherwise(0)).alias("linux"),
).collect()[0]

TAUX_WINDOWS = 100.0 * compte_plateformes["windows"] / NB_JEUX
TAUX_MAC     = 100.0 * compte_plateformes["mac"]     / NB_JEUX
TAUX_LINUX   = 100.0 * compte_plateformes["linux"]   / NB_JEUX

df_plateformes_viz = spark.createDataFrame([
    Row(plateforme="Windows", nombre_jeux=int(compte_plateformes["windows"]), pct_catalogue=round(TAUX_WINDOWS, 2)),
    Row(plateforme="Mac",     nombre_jeux=int(compte_plateformes["mac"]),     pct_catalogue=round(TAUX_MAC, 2)),
    Row(plateforme="Linux",   nombre_jeux=int(compte_plateformes["linux"]),   pct_catalogue=round(TAUX_LINUX, 2)),
])

display(df_plateformes_viz)

print(f"Total de jeux : {NB_JEUX:,}")
print(f"  Windows : {compte_plateformes['windows']:>7,} ({TAUX_WINDOWS:.2f} %)")
print(f"  Mac     : {compte_plateformes['mac']:>7,} ({TAUX_MAC:.2f} %)")
print(f"  Linux   : {compte_plateformes['linux']:>7,} ({TAUX_LINUX:.2f} %)")

plateforme,nombre_jeux,pct_catalogue
Windows,54713,99.98
Mac,12612,23.05
Linux,8388,15.33


Total de jeux : 54,725
  Windows :  54,713 (99.98 %)
  Mac     :  12,612 (23.05 %)
  Linux   :   8,388 (15.33 %)


In [0]:
# --- 5.2 Combinaisons exactes de plateformes ----------------------------------
# "2 plateformes" est ambigu : Windows+Mac et Windows+Linux ne racontent pas
# la même histoire. On détaille donc la combinaison réelle.
# Visualisation Databricks conseillée : pie chart (combinaison / nombre_jeux)

combinaisons = (
    df_games
    .withColumn(
        "combinaison",
        F.concat_ws(" + ", F.array_remove(F.array(
            F.when(F.col("windows"), F.lit("Windows")).otherwise(F.lit("")),
            F.when(F.col("mac"),     F.lit("Mac")).otherwise(F.lit("")),
            F.when(F.col("linux"),   F.lit("Linux")).otherwise(F.lit("")),
        ), "")),
    )
    .withColumn("combinaison",
                F.when(F.col("combinaison") == "", "Aucune plateforme déclarée")
                 .otherwise(F.col("combinaison")))
    .groupBy("combinaison", "nb_plateformes")
    .agg(
        F.count("*").alias("nombre_jeux"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
        F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen"),
    )
    .withColumn("pct_catalogue", F.round(100.0 * F.col("nombre_jeux") / NB_JEUX, 2))
    .orderBy(F.desc("nombre_jeux"))
)

display(combinaisons)

combinaison,nb_plateformes,nombre_jeux,prix_moyen_usd,ratio_moyen,pct_catalogue
Windows,1,40472,7.61,72.31,73.96
Windows + Mac + Linux,3,6746,7.95,78.32,12.33
Windows + Mac,2,5857,7.21,77.17,10.7
Windows + Linux,2,1638,6.5,78.19,2.99
Mac,1,8,27.62,43.3,0.01
Linux,1,3,10.0,70.49,0.01
Mac + Linux,2,1,4.99,84.13,0.0


In [0]:
# --- 5.3 Prix et qualité par plateforme ---------------------------------------
# Un SEUL dataframe en format long : indispensable pour produire un graphique
# comparatif dans Databricks (trois dataframes séparés ne sont pas graphables).

def profil_plateforme(nom, condition):
    # Le libellé est ajouté dans un select() APRÈS l'agrégation : un littéral
    # placé directement dans agg() n'est pas une expression d'agrégation.
    return (
        df_games.filter(condition)
        .agg(
            F.count("*").alias("nombre_jeux"),
            F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
            F.expr("percentile_approx(price_usd, 0.5)").alias("prix_median_usd"),
            F.round(F.avg("ratio_positif"), 2).alias("ratio_positif_moyen"),
            F.round(F.avg("total_reviews"), 0).alias("avis_moyen"),
            F.round(F.avg("owners_mid"), 0).alias("possesseurs_moyen"),
        )
        .select(F.lit(nom).alias("plateforme"), "nombre_jeux", "prix_moyen_usd",
                "prix_median_usd", "ratio_positif_moyen", "avis_moyen", "possesseurs_moyen")
    )

comparaison_plateformes = (
    profil_plateforme("Windows", F.col("windows"))
    .union(profil_plateforme("Mac",   F.col("mac")))
    .union(profil_plateforme("Linux", F.col("linux")))
)

display(comparaison_plateformes)

print("⚠️ BIAIS DE SÉLECTION à énoncer : les jeux portés sur Mac et Linux affichent")
print("de meilleurs indicateurs, mais ce n'est PAS le portage qui les rend meilleurs.")
print("Un studio ne finance un portage que pour un jeu qui marche déjà. La causalité")
print("va donc du succès vers le portage, et non l'inverse.")

plateforme,nombre_jeux,prix_moyen_usd,prix_median_usd,ratio_positif_moyen,avis_moyen,possesseurs_moyen
Windows,54713,7.58,4.99,73.75,1737.0,135092.0
Mac,12612,7.62,4.99,77.76,3289.0,237770.0
Linux,8388,7.67,4.99,78.29,3937.0,283715.0


⚠️ BIAIS DE SÉLECTION à énoncer : les jeux portés sur Mac et Linux affichent
de meilleurs indicateurs, mais ce n'est PAS le portage qui les rend meilleurs.
Un studio ne finance un portage que pour un jeu qui marche déjà. La causalité
va donc du succès vers le portage, et non l'inverse.


In [0]:
# --- 5.4 Certains genres sont-ils préférentiellement portés ? ------------------
# `pct_windows` vaut ~100 % pour tous les genres : cette métrique n'apprend rien.
# On raisonne donc en INDICE DE SUR-REPRÉSENTATION par rapport au taux de portage
# moyen du catalogue (indice > 1 = genre davantage porté que la moyenne).
# Visualisation Databricks conseillée : bar chart (genre_simple / index_mac, index_linux)

portage_par_genre = (
    df_genres
    .groupBy("genre_simple")
    .agg(
        F.countDistinct("appid").alias("nombre_jeux"),
        F.countDistinct(F.when(F.col("mac"), F.col("appid"))).alias("sur_mac"),
        F.countDistinct(F.when(F.col("linux"), F.col("appid"))).alias("sur_linux"),
    )
    .filter(F.col("nombre_jeux") >= 50)
    .withColumn("pct_mac",   F.round(100.0 * F.col("sur_mac")   / F.col("nombre_jeux"), 1))
    .withColumn("pct_linux", F.round(100.0 * F.col("sur_linux") / F.col("nombre_jeux"), 1))
    .withColumn("index_mac",   F.round(F.col("pct_mac")   / F.lit(TAUX_MAC), 2))
    .withColumn("index_linux", F.round(F.col("pct_linux") / F.lit(TAUX_LINUX), 2))
    .select("genre_simple", "nombre_jeux", "pct_mac", "index_mac", "pct_linux", "index_linux")
    .orderBy(F.desc("index_mac"))
)

display(portage_par_genre)

print(f"Taux de portage de référence — Mac : {TAUX_MAC:.1f} %  |  Linux : {TAUX_LINUX:.1f} %")
print("Lecture : un index de 1,20 signifie que le genre est porté sur Mac 1,2 fois")
print("plus souvent que la moyenne du catalogue. Un index < 1 signale un genre délaissé.")
print("Les écarts restent modérés une fois les logiciels écartés : le genre n'explique")
print("qu'une petite part de la décision de portage.")

In [0]:
# --- 5.5 Les jeux multi-plateformes les plus populaires -----------------------
# ⚠️ Le comptage et le classement sont deux opérations DISTINCTES : appliquer
# .count() à un dataframe déjà limité à 20 lignes renverrait toujours 20.

jeux_3_plateformes = df_games.filter(
    F.col("windows") & F.col("mac") & F.col("linux")
)

nb_3_plateformes = jeux_3_plateformes.count()

print(f"Jeux disponibles sur les 3 plateformes : {nb_3_plateformes:,} "
      f"({100.0 * nb_3_plateformes / NB_JEUX:.2f} % du catalogue)")

display(
    jeux_3_plateformes
    .select("name", "publisher", "release_year", "price_usd",
            "positive", "negative", "total_reviews", "ratio_positif", "owners_mid")
    .orderBy(F.desc("total_reviews"))
    .limit(20)
)

Jeux disponibles sur les 3 plateformes : 6,746 (12.33 % du catalogue)


name,publisher,release_year,price_usd,positive,negative,total_reviews,ratio_positif,owners_mid
Counter-Strike: Global Offensive,Valve,2012,0.0,5943345,787093,6730438,88.31,7.5E7
Dota 2,Valve,2013,0.0,1534895,317916,1852811,82.84,3.5E8
Terraria,Re-Logic,2011,9.99,1014711,22380,1037091,97.84,3.5E7
Team Fortress 2,Valve,2007,0.0,846407,57423,903830,93.65,7.5E7
Garry's Mod,Valve,2006,9.99,861240,29998,891238,96.63,3.5E7
Left 4 Dead 2,Valve,2009,9.99,643836,16828,660664,97.45,3.5E7
Euro Truck Simulator 2,SCS Software,2012,19.99,572368,15615,587983,97.34,1.5E7
Stardew Valley,ConcernedApe,2016,14.99,497558,9283,506841,98.17,1.5E7
Unturned,Smartly Dressed Games,2017,0.0,451671,42098,493769,91.47,3.5E7
Don't Starve Together,Klei Entertainment,2016,14.99,340987,13943,354930,96.07,1.5E7


---
# 6. Facteurs de succès — la question centrale

> *« Understand what factors affect the popularity or sales of a video game. »*

Les parties 3 à 5 décrivent le marché. Cette partie **croise les variables** pour
identifier ce qui distingue un jeu qui marche d'un jeu qui ne marche pas.

**Variable cible** : `owners_mid`, le milieu de la fourchette de possesseurs SteamSpy —
le meilleur proxy de ventes disponible dans ce dataset.

⚠️ **Corrélation n'est pas causalité.** Aucune des relations ci-dessous n'établit un lien
de cause à effet : un jeu traduit en 20 langues et porté sur 3 systèmes est d'abord
un jeu sur lequel un éditeur a déjà décidé d'investir.

⚠️ **`owners` est une variable en paliers, et cela change la lecture.** SteamSpy ne
publie pas un nombre mais une fourchette (`0 .. 20 000`, `20 000 .. 50 000`,
`500 000 .. 1 000 000`…). Le milieu de fourchette ne prend donc qu'une poignée de
valeurs : 10 000, 35 000, 75 000, 350 000… Conséquence directe : **la médiane des
possesseurs retombe presque toujours sur le même palier** et paraît plate, même quand
la distribution se déplace nettement. Chaque tableau de cette partie affiche donc la
**moyenne** et la **médiane du nombre d'avis** à côté de la médiane des possesseurs —
ce sont elles qui portent l'information.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import pandas as pd

VARIABLES = [
    "price_usd",       # prix courant
    "owners_mid",      # proxy de ventes (cible)
    "total_reviews",   # visibilité
    "ratio_positif",   # qualité perçue
    "ccu",             # joueurs simultanés
    "nb_langues",      # effort de localisation
    "nb_plateformes",  # effort de portage
    "nb_categories",   # richesse fonctionnelle Steam
    "release_year",    # ancienneté
]

df_corr = (
    df_games.select(*[F.col(c).cast("double").alias(c) for c in VARIABLES])
            .na.drop()
)

nb_obs = df_corr.count()
print(f"Observations complètes utilisées : {nb_obs:,} / {NB_JEUX:,} "
      f"({100.0 * nb_obs / NB_JEUX:.1f} %)")

assembleur = VectorAssembler(inputCols=VARIABLES, outputCol="features",
                             handleInvalid="skip")   # sécurité : écarte tout NaN résiduel
matrice = Correlation.corr(assembleur.transform(df_corr), "features", "pearson").head()[0]

pdf_corr = (
    pd.DataFrame(matrice.toArray(), index=VARIABLES, columns=VARIABLES)
    .round(3)
    .reset_index()
    .rename(columns={"index": "variable"})
)

display(spark.createDataFrame(pdf_corr))

Observations complètes utilisées : 54,466 / 54,725 (99.5 %)


variable,price_usd,owners_mid,total_reviews,ratio_positif,ccu,nb_langues,nb_plateformes,nb_categories,release_year
price_usd,1.0,0.034,0.045,0.066,0.021,0.125,0.003,0.185,0.036
owners_mid,0.034,1.0,0.578,0.024,0.774,0.089,0.036,0.094,-0.08
total_reviews,0.045,0.578,1.0,0.024,0.814,0.086,0.028,0.086,-0.048
ratio_positif,0.066,0.024,0.024,1.0,0.009,0.051,0.097,0.111,0.092
ccu,0.021,0.774,0.814,0.009,1.0,0.058,0.016,0.046,-0.017
nb_langues,0.125,0.089,0.086,0.051,0.058,1.0,0.091,0.196,0.024
nb_plateformes,0.003,0.036,0.028,0.097,0.016,0.091,1.0,0.195,-0.172
nb_categories,0.185,0.094,0.086,0.111,0.046,0.196,0.195,1.0,-0.125
release_year,0.036,-0.08,-0.048,0.092,-0.017,0.024,-0.172,-0.125,1.0


In [0]:
# Lecture directe : ce qui est le plus lié au nombre de possesseurs.
correlations_cible = (
    pdf_corr.set_index("variable")["owners_mid"]
    .drop("owners_mid")
    .sort_values(key=abs, ascending=False)
)

print("Corrélation de Pearson avec `owners_mid` (proxy de ventes)")
print("-" * 58)
for variable, valeur in correlations_cible.items():
    force = "forte" if abs(valeur) >= 0.5 else ("modérée" if abs(valeur) >= 0.2 else "faible")
    sens = "+" if valeur >= 0 else "−"
    print(f"  {variable:<16} {valeur:>7.3f}   ({sens} / {force})")

print("\nRéserve : Pearson ne capte que les relations LINÉAIRES. Les possesseurs et")
print("les avis suivent des distributions très asymétriques (quelques blockbusters,")
print("une longue traîne). Les analyses par segment ci-dessous sont plus parlantes.")

Corrélation de Pearson avec `owners_mid` (proxy de ventes)
----------------------------------------------------------
  ccu                0.774   (+ / forte)
  total_reviews      0.578   (+ / forte)
  nb_categories      0.094   (+ / faible)
  nb_langues         0.089   (+ / faible)
  release_year      -0.080   (− / faible)
  nb_plateformes     0.036   (+ / faible)
  price_usd          0.034   (+ / faible)
  ratio_positif      0.024   (+ / faible)

Réserve : Pearson ne capte que les relations LINÉAIRES. Les possesseurs et
les avis suivent des distributions très asymétriques (quelques blockbusters,
une longue traîne). Les analyses par segment ci-dessous sont plus parlantes.


In [0]:
# --- 6.1 La localisation paie-t-elle ? ----------------------------------------
# Visualisation Databricks conseillée : bar chart (tranche_langues / possesseurs_moyen)
# Lire la MOYENNE et la médiane d'avis : la médiane des possesseurs est en paliers (cf. intro).

ORDRE_LANGUES = ["1 langue", "2-3 langues", "4-6 langues", "7-12 langues", "13 langues et +"]

display(
    df_games
    .filter(F.col("nb_langues").isNotNull() & F.col("owners_mid").isNotNull())
    .withColumn(
        "tranche_langues",
        F.when(F.col("nb_langues") == 1, "1 langue")
         .when(F.col("nb_langues") <= 3, "2-3 langues")
         .when(F.col("nb_langues") <= 6, "4-6 langues")
         .when(F.col("nb_langues") <= 12, "7-12 langues")
         .otherwise("13 langues et +"),
    )
    .groupBy("tranche_langues")
    .agg(
        F.count("*").alias("nombre_jeux"),
        F.round(F.avg("owners_mid"), 0).alias("possesseurs_moyen"),
        F.expr("percentile_approx(owners_mid, 0.5)").alias("possesseurs_median"),
        F.expr("percentile_approx(total_reviews, 0.5)").alias("avis_median"),
        F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
    )
    .withColumn("ordre", F.array_position(
        F.array(*[F.lit(x) for x in ORDRE_LANGUES]), F.col("tranche_langues")))
    .orderBy("ordre")
    .drop("ordre")
)

tranche_langues,nombre_jeux,possesseurs_moyen,possesseurs_median,avis_median,ratio_moyen,prix_moyen_usd
1 langue,29128,48614.0,10000.0,16,72.49,6.03
2-3 langues,10817,50240.0,10000.0,23,73.69,7.18
4-6 langues,5580,166123.0,10000.0,85,75.26,10.07
7-12 langues,6309,362435.0,35000.0,199,76.92,12.14
13 langues et +,2881,770057.0,10000.0,96,76.76,9.96


In [0]:
# --- 6.2 Le portage multi-plateforme paie-t-il ? ------------------------------
# Visualisation Databricks conseillée : bar chart (nb_plateformes / possesseurs_moyen)

display(
    df_games
    .filter(F.col("owners_mid").isNotNull())
    .groupBy("nb_plateformes")
    .agg(
        F.count("*").alias("nombre_jeux"),
        F.round(F.avg("owners_mid"), 0).alias("possesseurs_moyen"),
        F.expr("percentile_approx(owners_mid, 0.5)").alias("possesseurs_median"),
        F.expr("percentile_approx(total_reviews, 0.5)").alias("avis_median"),
        F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
    )
    .orderBy("nb_plateformes")
)

print("Rappel du biais de sélection (cf. 5.3) : le portage est une CONSÉQUENCE du succès")
print("autant qu'une cause possible. Ce tableau mesure une association, pas un effet.")
print("La médiane des possesseurs est identique d'un palier à l'autre : c'est l'effet")
print("des paliers de `owners`. La médiane d'avis, elle, discrimine nettement.")

nb_plateformes,nombre_jeux,possesseurs_moyen,possesseurs_median,avis_median,ratio_moyen,prix_moyen_usd
1,40483,103335.0,10000.0,21,72.3,7.62
2,7496,138558.0,10000.0,37,77.39,7.06
3,6746,321637.0,10000.0,80,78.32,7.95


Rappel du biais de sélection (cf. 5.3) : le portage est une CONSÉQUENCE du succès
autant qu'une cause possible. Ce tableau mesure une association, pas un effet.
La médiane des possesseurs est identique d'un palier à l'autre : c'est l'effet
des paliers de `owners`. La médiane d'avis, elle, discrimine nettement.


In [0]:
# --- 6.3 Le prix influence-t-il la diffusion ? --------------------------------
# Visualisation Databricks conseillée : bar chart (segment_prix / revenu_moyen_musd)

ORDRE_SEGMENTS = ["Free-to-play", "Moins de 10 $", "10-30 $", "30-60 $", "60 $ et +"]

display(
    df_games
    .filter(F.col("price_usd").isNotNull() & F.col("owners_mid").isNotNull())
    .withColumn(
        "segment_prix",
        F.when(F.col("price_usd") == 0, "Free-to-play")
         .when(F.col("price_usd") < 10, "Moins de 10 $")
         .when(F.col("price_usd") < 30, "10-30 $")
         .when(F.col("price_usd") < 60, "30-60 $")
         .otherwise("60 $ et +"),
    )
    .groupBy("segment_prix")
    .agg(
        F.count("*").alias("nombre_jeux"),
        F.round(F.avg("owners_mid"), 0).alias("possesseurs_moyen"),
        F.expr("percentile_approx(owners_mid, 0.5)").alias("possesseurs_median"),
        F.expr("percentile_approx(total_reviews, 0.5)").alias("avis_median"),
        F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen"),
        F.round(F.avg(F.col("price_usd") * F.col("owners_mid")) / 1e6, 3).alias("revenu_moyen_musd"),
    )
    .withColumn("pct_catalogue", F.round(100.0 * F.col("nombre_jeux") / NB_JEUX, 2))
    .withColumn("ordre", F.array_position(
        F.array(*[F.lit(x) for x in ORDRE_SEGMENTS]), F.col("segment_prix")))
    .orderBy("ordre")
    .drop("ordre")
)

segment_prix,nombre_jeux,possesseurs_moyen,possesseurs_median,avis_median,ratio_moyen,revenu_moyen_musd,pct_catalogue
Free-to-play,7411,326772.0,10000.0,51,69.75,0.0,13.54
Moins de 10 $,35660,57922.0,10000.0,17,73.52,0.362,65.16
10-30 $,10629,210068.0,10000.0,99,76.93,4.306,19.42
30-60 $,962,680442.0,35000.0,937,77.55,32.491,1.76
60 $ et +,63,271429.0,10000.0,11,71.76,23.122,0.12


In [0]:
# --- 6.4 Quelles fonctionnalités Steam accompagnent le succès ? ---------------
# Pour chaque fonctionnalité, on compare les jeux qui la déclarent au catalogue entier.
# Visualisation Databricks conseillée : bar chart (categorie / index_moyenne)

reference = df_games.filter(F.col("owners_mid").isNotNull()).agg(
    F.expr("percentile_approx(owners_mid, 0.5)").alias("mediane"),
    F.avg("owners_mid").alias("moyenne"),
).collect()[0]

print(f"Référence catalogue — possesseurs médians : {reference['mediane']:,.0f}")
print(f"                      possesseurs moyens  : {reference['moyenne']:,.0f}")

impact_fonctionnalites = (
    df_games.filter(F.col("owners_mid").isNotNull())
    .select("appid", "owners_mid", "total_reviews", "ratio_positif",
            F.explode(F.col("categories")).alias("categorie"))
    .groupBy("categorie")
    .agg(
        F.countDistinct("appid").alias("nombre_jeux"),
        F.round(F.avg("owners_mid"), 0).alias("possesseurs_moyen"),
        F.expr("percentile_approx(owners_mid, 0.5)").alias("possesseurs_median"),
        F.expr("percentile_approx(total_reviews, 0.5)").alias("avis_median"),
        F.round(F.avg("ratio_positif"), 2).alias("ratio_moyen"),
    )
    .filter(F.col("nombre_jeux") >= 200)
    .withColumn("index_moyenne",
                F.round(F.col("possesseurs_moyen") / F.lit(float(reference["moyenne"])), 2))
    .withColumn("index_mediane",
                F.round(F.col("possesseurs_median") / F.lit(float(reference["mediane"])), 2))
    .orderBy(F.desc("index_moyenne"))
    .limit(25)
)

display(impact_fonctionnalites)

print("\nLecture : index_moyenne = 2,0 signifie que les jeux déclarant cette")
print("fonctionnalité ont une moyenne de possesseurs deux fois supérieure au catalogue.")
print("L'index sur la MÉDIANE est à lire avec prudence : `owners` étant une variable")
print("en paliers, il ne prend que des valeurs très rondes (1,0 / 3,5 / 7,5 / 35,0).")
print("\n⚠️ Ces fonctionnalités ne sont pas des leviers. Remote Play on Phone/Tablet,")
print("MMO ou In-App Purchases sont des MARQUEURS de grosses productions : elles")
print("accompagnent le succès, elles ne le fabriquent pas. Le sens de la causalité")
print("ne peut pas être établi avec ces données.")

Référence catalogue — possesseurs médians : 10,000
                      possesseurs moyens  : 135,070


categorie,nombre_jeux,possesseurs_moyen,possesseurs_median,avis_median,ratio_moyen,index_moyenne,index_mediane
Remote Play on Tablet,868,2349510.0,350000.0,5775,86.73,17.39,35.0
Remote Play on Phone,714,2049146.0,350000.0,3348,86.98,15.17,35.0
In-App Purchases,1323,1348723.0,75000.0,367,67.18,9.99,7.5
Commentary available,204,1118652.0,35000.0,254,75.73,8.28,3.5
Steam Workshop,1572,997861.0,35000.0,369,80.25,7.39,3.5
MMO,786,918842.0,75000.0,277,63.21,6.8,7.5
LAN Co-op,409,762861.0,10000.0,52,73.21,5.65,1.0
Remote Play on TV,1941,713689.0,75000.0,532,81.46,5.28,7.5
Online Co-op,2980,645247.0,35000.0,133,72.08,4.78,3.5
LAN PvP,474,601962.0,10000.0,40,73.51,4.46,1.0



Lecture : index_moyenne = 2,0 signifie que les jeux déclarant cette
fonctionnalité ont une moyenne de possesseurs deux fois supérieure au catalogue.
L'index sur la MÉDIANE est à lire avec prudence : `owners` étant une variable
en paliers, il ne prend que des valeurs très rondes (1,0 / 3,5 / 7,5 / 35,0).

⚠️ Ces fonctionnalités ne sont pas des leviers. Remote Play on Phone/Tablet,
MMO ou In-App Purchases sont des MARQUEURS de grosses productions : elles
accompagnent le succès, elles ne le fabriquent pas. Le sens de la causalité
ne peut pas être établi avec ces données.


In [0]:
# --- 6.5 Qualité et diffusion vont-elles de pair ? ---------------------------
# Visualisation Databricks conseillée : bar chart (tranche_qualite / avis_median)

ORDRE_QUALITE = ["Faible (< 50 %)", "Moyen (50-70 %)", "Bon (70-80 %)",
                 "Très bon (80-90 %)", "Excellent (90 %+)"]

display(
    df_games
    .filter((F.col("total_reviews") >= 50) & F.col("owners_mid").isNotNull())
    .withColumn(
        "tranche_qualite",
        F.when(F.col("ratio_positif") >= 90, "Excellent (90 %+)")
         .when(F.col("ratio_positif") >= 80, "Très bon (80-90 %)")
         .when(F.col("ratio_positif") >= 70, "Bon (70-80 %)")
         .when(F.col("ratio_positif") >= 50, "Moyen (50-70 %)")
         .otherwise("Faible (< 50 %)"),
    )
    .groupBy("tranche_qualite")
    .agg(
        F.count("*").alias("nombre_jeux"),
        F.round(F.avg("owners_mid"), 0).alias("possesseurs_moyen"),
        F.expr("percentile_approx(owners_mid, 0.5)").alias("possesseurs_median"),
        F.expr("percentile_approx(total_reviews, 0.5)").alias("avis_median"),
        F.round(F.avg("price_usd"), 2).alias("prix_moyen_usd"),
        F.round(F.avg("ccu"), 0).alias("joueurs_simultanes_moyen"),
    )
    .withColumn("ordre", F.array_position(
        F.array(*[F.lit(x) for x in ORDRE_QUALITE]), F.col("tranche_qualite")))
    .orderBy("ordre")
    .drop("ordre")
)

print("Trois régimes différents dans le même tableau :")
print("  • la MÉDIANE des possesseurs ne bouge pas d'une tranche à l'autre : effet des")
print("    paliers de `owners`. Le jeu médian reste confidentiel quelle que soit sa note.")
print("  • la MOYENNE des possesseurs, elle, est multipliée par près de 4 entre les jeux")
print("    mal notés et les mieux notés. L'effet est donc réel, mais il est porté par")
print("    les gros succès, pas par le jeu médian.")
print("  • le PRIX pratiqué et le VOLUME D'AVIS progressent régulièrement, tranche")
print("    après tranche.")
print("\nLa corrélation linéaire ratio_positif × owners_mid reste pourtant quasi nulle")
print("(r ≈ 0,02) : c'est la signature d'une relation NON LINÉAIRE, concentrée dans la")
print("queue de la distribution. Même mécanisme que pour la localisation (§6.1) — et")
print("bonne illustration de pourquoi un coefficient de Pearson seul aurait fait")
print("conclure, à tort, à une absence de relation.")

---
# 7. Synthèse et recommandations

La cellule suivante **génère la synthèse à partir des dataframes**, et non à partir de
chiffres recopiés à la main : elle ne peut donc pas diverger des tableaux du notebook.

In [0]:
# --- 7.1 Synthèse chiffrée, générée depuis les données ------------------------

def top1(df, col_libelle, col_valeur):
    ligne = df.orderBy(F.desc(col_valeur)).first()
    return ligne[col_libelle], ligne[col_valeur]

editeur_top, editeur_nb   = top1(df_publishers, "publisher", "nombre_jeux")
genre_top, genre_nb       = top1(top_genres, "genre_simple", "nombre_jeux")
genre_qualite, genre_ratio = top1(qualite_genres, "genre_simple", "ratio_pondere")
genre_revenu, genre_musd  = top1(revenu_par_genre, "genre_simple", "revenu_total_musd")
annee_top = max(annees, key=annees.get) if annees else None

print("=" * 68)
print(" SYNTHÈSE — CE QUE LES DONNÉES MONTRENT")
print("=" * 68)

print("\n■ VOLUME ET CONCURRENCE")
print(f"  • Catalogue analysé : {NB_JEUX:,} jeux, sur {NB_LIGNES:,} entrées Steam")
print(f"    ({nb_logiciels:,} logiciels et {nb_hors_type:,} entrée(s) hors type `game` écartés).")
print(f"  • Éditeur le plus prolifique : {editeur_top} ({editeur_nb:,} jeux).")
print(f"  • {nb_editeurs:,} éditeurs distincts : marché atomisé, dominé en volume")
print(f"    par des studios à forte cadence de publication.")
if annee_top:
    print(f"  • Année record de sorties : {annee_top} ({annees[annee_top]:,} jeux).")

print("\n■ QUALITÉ")
print(f"  • Ratio positif moyen (par jeu)    : {synthese_macro['ratio_moyen']:.2f} %")
print(f"  • Ratio positif pondéré (par avis) : {ratio_pondere:.2f} %")
print(f"  • Genre le mieux noté (pondéré)    : {genre_qualite} ({genre_ratio:.2f} %)")

print("\n■ PRIX")
print(f"  • Prix moyen : {stats_prix['prix_moyen']:.2f} $  |  médian : {stats_prix['prix_median']:.2f} $")
print(f"  • Gratuits   : {stats_prix['jeux_gratuits']:,} ({100.0*stats_prix['jeux_gratuits']/NB_JEUX:.1f} %)")
print(f"  • En promotion à l'instant T : {stats_promo['jeux_en_promo']:,} "
      f"({100.0*stats_promo['jeux_en_promo']/NB_JEUX:.1f} %), remise moyenne {stats_promo['remise_moyenne_pct']:.0f} %")

print("\n■ GENRES")
print(f"  • Genre le plus représenté : {genre_top} ({genre_nb:,} jeux)")
print(f"  • Genre au revenu estimé le plus élevé : {genre_revenu} ({genre_musd:,.0f} M$)")
print(f"  • Un jeu déclare en moyenne {nb_occurrences / nb_jeux_avec_genre:.1f} genres.")

print("\n■ PLATEFORMES")
print(f"  • Windows : {TAUX_WINDOWS:.2f} %  |  Mac : {TAUX_MAC:.2f} %  |  Linux : {TAUX_LINUX:.2f} %")
print(f"  • Jeux disponibles sur les 3 systèmes : {nb_3_plateformes:,} "
      f"({100.0*nb_3_plateformes/NB_JEUX:.2f} %)")

print("\n■ CLASSIFICATION PAR ÂGE")
print(f"  • Interdits aux moins de 18 ans : {nb_18:,} ({100.0*nb_18/NB_JEUX:.2f} %)")
print(f"  • Chiffre à lire comme une BORNE BASSE : le champ `required_age` est toujours")
print(f"    présent, mais il vaut 0 pour {100.0*(NB_JEUX-nb_age_positif)/NB_JEUX:.1f} % des jeux. Steam n'impose")
print(f"    une classification que pour certains contenus : la déclaration reste")
print(f"    à la main de l'éditeur et n'est pas systématique.")

print("\n■ FACTEURS ASSOCIÉS AU SUCCÈS (corrélation avec les possesseurs estimés)")
AUTRES_MESURES_DE_POPULARITE = {"total_reviews", "ccu"}
for variable, valeur in correlations_cible.items():
    note = "   <- autre mesure de popularité" if variable in AUTRES_MESURES_DE_POPULARITE else ""
    print(f"  • {variable:<16} r = {valeur:+.3f}{note}")
print("  Les deux seules corrélations fortes portent sur d'autres mesures de la même")
print("  chose : un jeu très possédé a beaucoup d'avis et de joueurs simultanés.")
print("  Toutes les variables ACTIONNABLES — langues, fonctionnalités, plateformes,")
print("  prix, qualité — restent sous 0,10. C'est LE résultat de la partie 6 :")
print("  aucun levier simple ne prédit le succès d'un jeu. Et aucune de ces")
print("  relations n'établit une causalité.")
print("=" * 68)

 SYNTHÈSE — CE QUE LES DONNÉES MONTRENT

■ VOLUME ET CONCURRENCE
  • Catalogue analysé : 54,725 jeux, sur 55,691 entrées Steam
    (965 logiciels et 1 entrée(s) hors type `game` écartés).
  • Éditeur le plus prolifique : Big Fish Games (422 jeux).
  • 29,497 éditeurs distincts : marché atomisé, dominé en volume
    par des studios à forte cadence de publication.
  • Année record de sorties : 2021 (8,722 jeux).

■ QUALITÉ
  • Ratio positif moyen (par jeu)    : 73.74 %
  • Ratio positif pondéré (par avis) : 85.88 %
  • Genre le mieux noté (pondéré)    : Indie (88.47 %)

■ PRIX
  • Prix moyen : 7.58 $  |  médian : 4.99 $
  • Gratuits   : 7,411 (13.5 %)
  • En promotion à l'instant T : 2,495 (4.6 %), remise moyenne 58 %

■ GENRES
  • Genre le plus représenté : Indie (39,681 jeux)
  • Genre au revenu estimé le plus élevé : Action (58,756 M$)
  • Un jeu déclare en moyenne 2.8 genres.

■ PLATEFORMES
  • Windows : 99.98 %  |  Mac : 23.05 %  |  Linux : 15.33 %
  • Jeux disponibles sur les 3 sys

## 7.2 Lecture analytique — ce que les chiffres impliquent

*Cette section interprète les résultats produits ci-dessus. Chaque affirmation renvoie
à une section du notebook.*

**Le marché est saturé et atomisé.** Près de 30 000 éditeurs se partagent le catalogue et
le plus prolifique d'entre eux ne pèse pas 1 % du volume (§3.1). La visibilité, et non la
capacité de production, est le goulot d'étranglement.

**La qualité perçue est élevée en moyenne, mais la moyenne cache tout.** Le ratio moyen par
jeu et le ratio pondéré par avis divergent de plus de douze points (§3.2) : les avis Steam
se concentrent massivement sur des titres bien notés. Un jeu médian ne recueille que
quelques dizaines d'avis — il n'est pas mal noté, il est invisible.

**Le marché est un marché de petits prix.** Le prix médian, la part de gratuits et le poids
de la tranche 0-5 $ (§3.4) situent Steam très loin du segment AAA. Plus frappant encore :
la tranche 60 $ et plus ne représente qu'une poignée de jeux (§6.3), et ces jeux recueillent
une médiane d'avis dérisoire. **Le point de prix AAA au-delà de 60 $ n'existe quasiment pas
sur Steam.**

**Le segment premium se situe entre 30 et 60 $.** C'est la seule tranche qui se détache sur
tous les indicateurs à la fois — possesseurs, avis, revenu moyen estimé (§6.3) — alors
qu'elle ne pèse que ~2 % du catalogue.

**Windows est un prérequis, pas un choix.** Le quasi-monopole (§5.1) rend l'arbitrage de
portage purement économique : Mac et Linux ne sont pas des marchés à conquérir mais des
options à évaluer, avec une réserve de causalité importante (§5.3, §6.2).

**Le genre discrimine peu la qualité.** Une fois les catégories logicielles écartées, les
neuf genres principaux tiennent dans un intervalle de six points de ratio pondéré (§4.3),
et le genre le plus représenté est aussi celui qui affiche le meilleur ratio. Seuls quelques
genres de niche décrochent nettement en bas de classement. Le genre est en revanche très
discriminant sur le **revenu estimé** (§4.5), où l'écart atteint un ordre de grandeur entre
le premier et le dernier des grands genres : c'est là qu'un arbitrage éditorial se joue,
pas sur la satisfaction.

**Aucun levier simple ne prédit le succès.** C'est le résultat central de la partie 6, et
il mérite d'être énoncé comme tel plutôt que subi. Les deux fortes corrélations avec le
nombre de possesseurs (joueurs simultanés, volume d'avis) sont d'autres mesures de la même
chose. Toutes les variables sur lesquelles un studio peut réellement agir — nombre de
langues, fonctionnalités Steam, portage, prix, qualité perçue — restent sous 0,10 de
corrélation. Le succès sur Steam n'est pas une recette : c'est une distribution à queue
très lourde, où une poignée de titres capte l'essentiel de l'attention.

**Ce qui bouge quand même, et par quel canal.** Trois effets réels apparaissent, et ils
partagent tous le même mécanisme : ils agissent sur la **queue haute** de la distribution,
jamais sur le jeu médian. C'est pourquoi les coefficients de corrélation restent faibles
alors que les écarts par segment sont considérables.

- La **localisation** multiplie par près de seize la moyenne des possesseurs entre un jeu
  monolingue et un jeu traduit en treize langues ou plus (§6.1) — alors que la *médiane*
  ne bouge pas. La localisation n'améliore pas le sort du jeu médian, elle accompagne les
  titres qui percent.
- La **qualité perçue** multiplie par près de quatre la moyenne des possesseurs entre les
  jeux mal notés et les mieux notés (§6.5), et fait progresser régulièrement le **prix
  pratiqué** et le **volume d'avis**. Mais là encore la médiane est plate, et la corrélation
  linéaire est quasi nulle (r ≈ 0,02). Bien noter son jeu ne sort pas de l'anonymat ; en
  revanche, parmi les jeux qui percent, les mieux notés percent beaucoup plus fort.
- Le **portage multi-plateforme** triple la moyenne des possesseurs entre un jeu Windows
  seul et un jeu disponible sur les trois systèmes (§6.2) — avec la réserve de causalité la
  plus forte du notebook : c'est le succès qui finance le portage autant que l'inverse.

Ces trois observations disent la même chose sous trois angles : sur Steam, **rien ne fait
sortir un jeu de l'anonymat de façon fiable**, mais plusieurs facteurs amplifient nettement
un succès déjà amorcé.

---

## 7.3 Recommandations pour Ubisoft

> ⚠️ **Statut de cette section : recommandations métier.**
> Contrairement aux parties 1 à 6, elle ne découle pas uniquement du dataset. Elle combine
> les constats ci-dessus avec des éléments qui ne sont **pas** mesurables dans ces données
> (coûts de production, budget marketing, valeur de marque, calendrier concurrentiel). Elle
> est présentée séparément pour que la frontière entre *ce que la donnée démontre* et *ce
> que l'analyste recommande* reste explicite.

**1. Prix — la recommandation la mieux étayée.** Viser la tranche **30-60 $** plutôt que le
seuil symbolique des 60 $. C'est la seule tranche qui se détache sur tous les indicateurs
(§6.3), et le segment au-delà de 60 $ est statistiquement marginal sur Steam. Un
positionnement AAA se défend donc — mais dans le haut de la fourchette 30-60 $, pas au-delà.

**2. Genre — arbitrer sur le revenu, pas sur la satisfaction.** Les écarts de qualité entre
genres de jeu sont trop faibles pour fonder un choix (§4.3), alors que les écarts de revenu
estimé sont d'un ordre de grandeur (§4.5). Le croisement volume × revenu × prix médian
(§4.2, §4.4, §4.5) est l'outil d'arbitrage ; la satisfaction moyenne du genre ne l'est pas.

**3. Plateformes.** Windows en priorité absolue (§5.1). Mac et Linux à évaluer *après*
validation commerciale, avec une réserve explicite : les meilleurs indicateurs des jeux
portés reflètent une sélection, pas un effet du portage (§5.3, §6.2).

**4. Localisation — un pari sur le haut de la distribution.** L'effet mesuré est réel mais
il ne concerne pas le jeu médian (§6.1). Pour un éditeur qui vise le succès de masse — ce
qui est le cas d'Ubisoft — c'est précisément la bonne moitié de la distribution : prévoir
la localisation dès la conception se justifie. Pour un studio indépendant, l'arbitrage
serait différent.

**5. Fonctionnalités Steam — des marqueurs, pas des leviers.** Les fonctionnalités associées
aux plus fortes diffusions (§6.4) sont celles des grosses productions : les intégrer ne
produit pas le succès, mais leur absence signale une production de moindre ambition à une
communauté qui sait les lire. À traiter comme un standard de marché à respecter, pas comme
un levier de croissance.

**6. Qualité au lancement — un amplificateur, pas un déclencheur.** Les jeux les mieux
notés affichent une moyenne de possesseurs quatre fois supérieure aux moins bien notés, un
prix pratiqué en hausse régulière et trois fois plus d'avis (§6.5) — mais leur médiane est
identique, et la corrélation linéaire est nulle. La qualité n'extrait donc pas un jeu de
l'anonymat ; elle démultiplie un succès qui a déjà commencé. Pour un éditeur qui dispose
d'une force de frappe marketing — ce qui est le cas d'Ubisoft — c'est précisément la
situation où l'investissement qualité paie : le lancement crée la visibilité, la qualité
détermine l'ampleur de ce qui suit.

---

## 7.4 Limites de l'analyse et pistes d'approfondissement

- **Photographie instantanée** : ni historique de prix, ni historique d'avis, ni suivi des
  jeux retirés du catalogue. Toute lecture temporelle (§3.3) porte sur les *survivants*, et
  la dernière année du dataset est tronquée.
- **`owners` est une variable en paliers**, pas une mesure continue. C'est la limite la plus
  structurante de la partie 6 : elle écrase les médianes et interdit toute analyse fine de
  la diffusion. Les revenus estimés (§4.5) sont des ordres de grandeur relatifs, jamais des
  montants.
- **Le champ `type` est inexploitable** sur ce dataset (`game` pour 55 690 entrées sur
  55 691, logiciels compris). La séparation jeux / logiciels repose donc sur une liste de
  genres construite à la main (§2.6) : c'est une heuristique, pas une vérité de terrain.
- **`required_age` est déclaratif** et vaut 0 dans la quasi-totalité des cas : les chiffres
  de classification par âge (§3.7) sont des bornes basses.
- **Corrélations linéaires uniquement** (§6), sur des distributions à queue très lourde où
  Pearson est peu adapté. Un coefficient de Spearman, ou une analyse sur les rangs, serait
  plus robuste.
- **Pistes** : exploiter le champ `tags` (plusieurs centaines de tags communautaires, bien
  plus fins que la vingtaine de genres officiels) ; analyser `short_description` en NLP ;
  modéliser `owners_mid` par une régression sur les variables de la partie 6 pour
  hiérarchiser les facteurs plutôt que de les observer un à un.